# Structured R1 — Participation Branch Ablation Study

This notebook performs the **participation-branch ablation study** on top of the previously selected **Structured R1** consolidation configuration.

Structured R1 is used as the fixed baseline because it provides both:

- a final binary `NORMAL` / `ANOMALOUS` prediction;
- explicit categorical assessments for participation, local temporal evidence, global temporal evidence, combined temporal evidence, semantic compatibility, and the decisive evidence dimension.

The full Structured R1 baseline uses the same 400-case development set:

- 100 NORMAL,
- 100 LAG,
- 100 WRONG PARTNER,
- 100 SILENT PARTNER.

Its thesis-reported performance is:

| Configuration | NORMAL | LAG | WRONG | SILENT | Accuracy |
|---|---:|---:|---:|---:|---:|
| **Full Structured R1** | **78/100** | **80/100** | **88/100** | **100/100** | **86.50%** |

## Participation evidence in Structured R1

In the full Structured R1 evidence packet, the participation branch contains two participant-level components:

1. **`speaks`**
   - a compact Boolean-style indicator derived from the participant's processed VAD evidence;
   - provides explicit grounding for whether meaningful speech is present.

2. **`filtered_turns`**
   - the cleaned participant-specific speaking-turn sequence;
   - derived from VAD intervals after the preprocessing and filtering used by the temporal pipeline.

The purpose of this notebook is to determine whether the participation branch is useful as a whole and whether one of these two components contributes more than the other.

## Controlled ablation design

The full Structured R1 setup is retained as the reference configuration:

```text
A0 — Full Structured R1
    participation = speaks + filtered_turns
```

Three controlled interventions are then evaluated.

### A1 — Remove the entire participation branch

Both participation components are removed:

```text
speaks          → removed
filtered_turns  → removed
```

The corresponding participation-specific prompt instructions are also removed.

This tests whether the participation branch provides useful information beyond the semantic and temporal branches.

### A1-S — Remove `speaks`, keep `filtered_turns`

```text
speaks          → removed
filtered_turns  → retained
```

This isolates the contribution of the detailed speaking-turn representation when the explicit speech-presence indicator is unavailable.

### A1-T — Keep `speaks`, remove `filtered_turns`

```text
speaks          → retained
filtered_turns  → removed
```

This tests whether the compact grounded participation indicator is sufficient without exposing the complete turn sequence to the reasoner.

## Experimental control

For every ablation, the notebook preserves the non-ablated parts of Structured R1:

- the same 400 development cases;
- the same Qwen2.5-Omni model;
- the same deterministic decoding settings;
- the same local temporal evidence;
- the same global temporal evidence;
- the same frozen NORMAL temporal reference;
- the same coarse semantic summaries;
- the same focused semantic summaries;
- the same Structured R1 output schema;
- the same final binary classification task.

Each intervention removes both the targeted evidence field and its corresponding prompt instructions so that the ablated model is not asked to reason about information that is no longer present.

## Main results

The final comparison produced by this notebook is:

| Configuration | NORMAL | LAG | WRONG | SILENT | Accuracy |
|---|---:|---:|---:|---:|---:|
| Full Structured R1 | 78/100 | 80/100 | 88/100 | 100/100 | 86.50% |
| No participation branch | 47/100 | 93/100 | 94/100 | 69/100 | 75.75% |
| Filtered turns only | 75/100 | 82/100 | 94/100 | 100/100 | 87.75% |
| **Speaks only** | **78/100** | **90/100** | **89/100** | **100/100** | **89.25%** |

The ablation reveals two important effects.

First, removing the participation branch entirely substantially reduces NORMAL preservation and Silent Partner detection, confirming that participation evidence contributes meaningfully to the unified reasoner.

Second, the compact `speaks` indicator is more useful than the more detailed `filtered_turns` representation in this consolidation setting. Retaining `speaks` while removing `filtered_turns` produces the strongest participation configuration, improving overall accuracy from **86.50% to 89.25%**.

This result motivates the later decision to retain **`speaks` only** as the participation representation for subsequent ablations.

The notebook also retains the original structured-assessment inspections for each ablation so that changes in classification behaviour can be compared with changes in the observable Structured R1 evidence assessments.

> **Reproducibility note:** every code cell, saved output, execution count, code-cell metadata field, evaluation table, and inspection result is preserved exactly as executed. Only this introductory Markdown cell has been rewritten for the public repository.

## 1. Install dependencies

Run this cell once in a fresh Colab runtime. Then select:

`Runtime → Restart session`

and continue from the next cell.

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece
!pip install -U qwen-omni-utils decord ffmpeg-python
!apt-get update -qq
!apt-get install -y -qq ffmpeg

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 152.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 148.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 77.9 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Mount Google Drive and configure artifact paths

The folder name still contains `1_2_3sec` for historical reasons. The finalized 400-case database is loaded from the exact path below.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
from collections import Counter
import copy
import json

import pandas as pd
from IPython.display import display


OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)


REFERENCE_BASE_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_base_statistics.json"
)


REFERENCE_SHIFT_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_global_shift_statistics.json"
)


FINAL_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_all_400_cases_with_"
        "temporal_and_semantic_summaries.json"
    )
)


MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

MODEL_PATH = Path(
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)


required_paths = {
    "Frozen base statistics": (
        REFERENCE_BASE_STATS_PATH
    ),

    "Frozen global-shift statistics": (
        REFERENCE_SHIFT_STATS_PATH
    ),

    "Final 400-case database": (
        FINAL_DATABASE_PATH
    ),

    "Local Qwen checkpoint": (
        MODEL_PATH
    ),
}


print("=" * 88)
print("ARTIFACT PATH AUDIT")
print("=" * 88)

for name, path in required_paths.items():

    print(
        f"{name}:",
        path,
    )

    print(
        "  exists:",
        path.exists(),
    )


assert OUT_DIR.exists(), (
    f"Project directory not found: {OUT_DIR}"
)

assert REFERENCE_BASE_STATS_PATH.exists(), (
    "Frozen base-statistics file not found: "
    f"{REFERENCE_BASE_STATS_PATH}"
)

assert REFERENCE_SHIFT_STATS_PATH.exists(), (
    "Frozen global-shift-statistics file not found: "
    f"{REFERENCE_SHIFT_STATS_PATH}"
)

assert FINAL_DATABASE_PATH.exists(), (
    "Final consolidation database not found: "
    f"{FINAL_DATABASE_PATH}"
)

assert MODEL_PATH.exists(), (
    "Local Qwen checkpoint not found: "
    f"{MODEL_PATH}"
)

Mounted at /content/drive
ARTIFACT PATH AUDIT
Frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json
  exists: True
Frozen global-shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json
  exists: True
Final 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
  exists: True
Local Qwen checkpoint: /content/drive/MyDrive/Qwen2.5-Omni-7B
  exists: True


## 3. Load the frozen NORMAL reference statistics

Both frozen JSON files contain separate reference profiles. This experiment deliberately extracts and retains only:

```text
reference_profile = NORMAL
```

The lag reference profiles are not included in the binary-only experiment state at this stage.

In [ ]:
# ============================================================
# LOAD FROZEN STATISTICS AND RETAIN NORMAL ONLY
# ============================================================

def load_json_list(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        list,
    ), (
        f"Expected a JSON list in {path}"
    )


    return data


frozen_base_records_all_profiles = (
    load_json_list(
        REFERENCE_BASE_STATS_PATH
    )
)


frozen_shift_records_all_profiles = (
    load_json_list(
        REFERENCE_SHIFT_STATS_PATH
    )
)


base_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_base_records_all_profiles
]


shift_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_shift_records_all_profiles
]


assert len(
    base_profiles
) == len(
    set(
        base_profiles
    )
)


assert len(
    shift_profiles
) == len(
    set(
        shift_profiles
    )
)


assert "NORMAL" in base_profiles
assert "NORMAL" in shift_profiles


normal_base_matches = [
    record

    for record
    in frozen_base_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


normal_shift_matches = [
    record

    for record
    in frozen_shift_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


assert len(
    normal_base_matches
) == 1


assert len(
    normal_shift_matches
) == 1


frozen_normal_base_statistics = copy.deepcopy(
    normal_base_matches[0]
)


frozen_normal_shift_statistics = copy.deepcopy(
    normal_shift_matches[0]
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# This is the only frozen reference object that should later
# be passed to the binary-only prompt constructor.
FROZEN_NORMAL_REFERENCE = {
    "base_statistics": (
        frozen_normal_base_statistics
    ),

    "global_shift_statistics": (
        frozen_normal_shift_statistics
    ),
}


normal_base_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_base_statistics.items()

    if key != "reference_profile"
])


normal_shift_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_shift_statistics.items()

    if key != "reference_profile"
])


print("=" * 88)
print("FROZEN REFERENCE FILES LOADED")
print("=" * 88)

print(
    "Profiles stored in base-statistics file:",
    base_profiles,
)

print(
    "Profiles stored in global-shift file:",
    shift_profiles,
)


print("\n" + "=" * 88)
print("FROZEN NORMAL BASE TEMPORAL STATISTICS")
print("=" * 88)

display(
    normal_base_display_df
)


print("\n" + "=" * 88)
print("FROZEN NORMAL GLOBAL-SHIFT STATISTICS")
print("=" * 88)

display(
    normal_shift_display_df
)


print(
    "\nOnly NORMAL frozen references retained:",
    list(
        FROZEN_NORMAL_REFERENCE.keys()
    ),
)

FROZEN REFERENCE FILES LOADED
Profiles stored in base-statistics file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']
Profiles stored in global-shift file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']

FROZEN NORMAL BASE TEMPORAL STATISTICS


,metric,value
0,num_original_A_turns,11.800000
1,num_original_B_turns,11.080000
2,num_removed_A_backchannels,1.740000
3,num_removed_B_backchannels,1.660000
4,num_filtered_A_turns,10.060000
5,num_filtered_B_turns,9.420000
6,num_offsets,5.760000
7,num_negative,2.040000
8,num_positive,3.720000
9,offset_mean,0.302128



FROZEN NORMAL GLOBAL-SHIFT STATISTICS


,metric,value
0,best_B_correction_shift_seconds__mean,-0.230000
1,best_B_correction_shift_seconds__median,-0.000000
2,estimated_B_lateness_seconds__mean,0.500000
3,estimated_B_lateness_seconds__median,0.000000
4,alignment_score_gain_vs_zero__mean,0.023376
5,alignment_score_gain_vs_zero__median,0.009105
6,best_num_bilateral_events__mean,11.940000
7,best_event_coverage__mean,0.595175



Only NORMAL frozen references retained: ['base_statistics', 'global_shift_statistics']


## 4. Load and audit the final 400-case database

This cell verifies:

- exactly 400 unique cases;
- exactly 100 cases per family;
- 100 `NORMAL` and 300 `ANOMALOUS` binary labels;
- participant-level `speaks` agrees with the final filtered VAD turns;
- every sample has all four coarse and all four focused summaries;
- no focused summary contains the legacy `speaks` field.

In [ ]:
# ============================================================
# LOAD FINAL DATABASE
# ============================================================

consolidation_cases = load_json_list(
    FINAL_DATABASE_PATH
)


assert len(
    consolidation_cases
) == 400


# ============================================================
# NORMALIZE CASE FAMILY
# ============================================================

def get_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        "Unknown case_variant: "
        f"{case.get('case_variant')}"
    )


# ============================================================
# GLOBAL COUNTS
# ============================================================

case_ids = [
    str(
        case[
            "case_id"
        ]
    )

    for case
    in consolidation_cases
]


assert len(
    case_ids
) == len(
    set(
        case_ids
    )
)


family_counts = Counter(
    get_case_family(
        case
    )

    for case
    in consolidation_cases
)


expected_family_counts = {
    "normal": 100,
    "wrong_partner": 100,
    "lag": 100,
    "silent_partner": 100,
}


assert dict(
    family_counts
) == expected_family_counts


binary_label_counts = Counter(
    str(
        case[
            "gold_binary_label"
        ]
    )

    for case
    in consolidation_cases
)


assert binary_label_counts[
    "NORMAL"
] == 100


assert binary_label_counts[
    "ANOMALOUS"
] == 300


lag_variant_counts = Counter(
    str(
        case.get(
            "case_variant"
        )
    )

    for case
    in consolidation_cases

    if get_case_family(
        case
    ) == "lag"
)


# ============================================================
# PARTICIPANT AND SEMANTIC AUDITS
# ============================================================

SEGMENT_NAMES = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]


num_participant_records_checked = 0
num_semantic_slots_checked = 0


for case in consolidation_cases:

    for (
        role,
        turns_key,
    ) in [
        (
            "participant_A",
            "participant_A_filtered_turns",
        ),

        (
            "participant_B",
            "participant_B_filtered_turns",
        ),
    ]:

        participant = case[
            role
        ]


        turns = case[
            turns_key
        ]


        assert isinstance(
            participant,
            dict,
        )


        assert isinstance(
            turns,
            list,
        )


        assert (
            "speaks"
            in participant
        )


        assert isinstance(
            participant[
                "speaks"
            ],
            bool,
        )


        assert (
            participant[
                "speaks"
            ]
            ==
            (
                len(
                    turns
                ) > 0
            )
        ), (
            "Participant-level speaks disagrees with "
            f'VAD turns in {case["case_id"]} / {role}'
        )


        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        for segment_name in SEGMENT_NAMES:

            segment_record = (
                participant_semantics[
                    segment_name
                ]
            )


            coarse_summary = (
                segment_record.get(
                    "coarse_summary"
                )
            )


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            assert isinstance(
                coarse_summary,
                dict,
            ), (
                "Missing coarse summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert isinstance(
                focused_summary,
                dict,
            ), (
                "Missing focused summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert (
                "speaks"
                not in focused_summary
            ), (
                "Legacy focused-summary speaks field found in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            num_semantic_slots_checked += 1


        num_participant_records_checked += 1


# ============================================================
# DISPLAY DATABASE COMPOSITION
# ============================================================

family_count_df = pd.DataFrame([
    {
        "case_family": family,
        "num_cases": count,
    }

    for family, count
    in sorted(
        family_counts.items()
    )
])


binary_count_df = pd.DataFrame([
    {
        "gold_binary_label": label,
        "num_cases": count,
    }

    for label, count
    in sorted(
        binary_label_counts.items()
    )
])


lag_variant_df = pd.DataFrame([
    {
        "case_variant": variant,
        "num_cases": count,
    }

    for variant, count
    in sorted(
        lag_variant_counts.items()
    )
])


print("=" * 88)
print("FINAL CONSOLIDATION DATABASE AUDIT PASSED")
print("=" * 88)

print(
    "Database:",
    FINAL_DATABASE_PATH,
)

print(
    "Total unique cases:",
    len(
        consolidation_cases
    ),
)

print(
    "Participant records checked:",
    num_participant_records_checked,
)

print(
    "Semantic participant-segment slots checked:",
    num_semantic_slots_checked,
)


print("\nCASE FAMILIES")

display(
    family_count_df
)


print("\nBINARY LABELS")

display(
    binary_count_df
)


print("\nLAG VARIANTS STORED IN THE FINAL DATABASE")

display(
    lag_variant_df
)

FINAL CONSOLIDATION DATABASE AUDIT PASSED
Database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Total unique cases: 400
Participant records checked: 800
Semantic participant-segment slots checked: 1600

CASE FAMILIES


,case_family,num_cases
0,lag,100
1,normal,100
2,silent_partner,100
3,wrong_partner,100



BINARY LABELS


,gold_binary_label,num_cases
0,ANOMALOUS,300
1,NORMAL,100



LAG VARIANTS STORED IN THE FINAL DATABASE


,case_variant,num_cases
0,lag_2sec,50
1,lag_3sec,50


## 5. Participant-level `speaks` overview

The field is derived exclusively from each sample's final filtered VAD turns.

In [ ]:
# ============================================================
# PARTICIPANT-LEVEL SPEAKS SUMMARY
# ============================================================

speaks_rows = []


for case in consolidation_cases:

    speaks_rows.append({
        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_id": (
            case[
                "case_id"
            ]
        ),

        "participant_A_speaks": (
            case[
                "participant_A"
            ][
                "speaks"
            ]
        ),

        "participant_B_speaks": (
            case[
                "participant_B"
            ][
                "speaks"
            ]
        ),

        "participant_A_num_turns": len(
            case[
                "participant_A_filtered_turns"
            ]
        ),

        "participant_B_num_turns": len(
            case[
                "participant_B_filtered_turns"
            ]
        ),
    })


speaks_df = pd.DataFrame(
    speaks_rows
)


speaks_summary_df = (
    speaks_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        participant_A_speaks_true=(
            "participant_A_speaks",
            "sum",
        ),

        participant_B_speaks_true=(
            "participant_B_speaks",
            "sum",
        ),
    )
)


speaks_summary_df[
    "participant_A_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_A_speaks_true"
    ]
)


speaks_summary_df[
    "participant_B_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_B_speaks_true"
    ]
)


speaks_summary_df = speaks_summary_df[
    [
        "case_family",
        "total_cases",

        "participant_A_speaks_true",
        "participant_A_speaks_false",

        "participant_B_speaks_true",
        "participant_B_speaks_false",
    ]
]


display(
    speaks_summary_df
)

,case_family,total_cases,participant_A_speaks_true,participant_A_speaks_false,participant_B_speaks_true,participant_B_speaks_false
0,lag,100,100,0,100,0
1,normal,100,100,0,100,0
2,silent_partner,100,100,0,0,100
3,wrong_partner,100,100,0,100,0


## 6. Interactive case inspection

The inspector supports all four families:

- `NORMAL`
- `WRONG_PARTNER`
- `LAG`
- `SILENT_PARTNER`

It can display participant metadata, VAD-derived `speaks`, filtered turns, temporal features, semantic summaries, and the complete sample JSON.

The direct function can also be used without widgets:

```python
inspect_case(
    family="normal",
    sample_index=0,
)
```

In [ ]:
# ============================================================
# INTERACTIVE DATABASE INSPECTOR
# ============================================================

import ipywidgets as widgets

from IPython.display import (
    clear_output,
    display,
)


try:

    from google.colab import output

    output.enable_custom_widget_manager()

except Exception:

    pass


# ============================================================
# SPLIT AND SORT CASES
# ============================================================

cases_by_family = {
    "normal": [],
    "wrong_partner": [],
    "lag": [],
    "silent_partner": [],
}


for case in consolidation_cases:

    cases_by_family[
        get_case_family(
            case
        )
    ].append(
        case
    )


for family in cases_by_family:

    cases_by_family[
        family
    ] = sorted(
        cases_by_family[
            family
        ],

        key=lambda case: str(
            case.get(
                "case_id",
                "",
            )
        ),
    )


# ============================================================
# DISPLAY HELPERS
# ============================================================

def participant_row(
    case,
    role,
):
    participant = case[
        role
    ]


    turns_key = (
        f"{role}_filtered_turns"
    )


    turns = case.get(
        turns_key,
        [],
    )


    return {
        "role": role,

        "conversation_id": (
            participant.get(
                "conversation_id"
            )
        ),

        "participant_id": (
            participant.get(
                "participant_id"
            )
        ),

        "speaks": (
            participant.get(
                "speaks"
            )
        ),

        "num_filtered_turns": len(
            turns
        ),

        "metadata_path": (
            participant.get(
                "metadata_path"
            )
        ),
    }


def display_semantics(
    case,
    role,
):
    print(
        "\n" + "-" * 88
    )

    print(
        f"{role.upper()} SEMANTIC SUMMARIES"
    )

    print(
        "-" * 88
    )


    participant_semantics = (
        case[
            "semantic_summaries"
        ][role]
    )


    for segment_name in SEGMENT_NAMES:

        segment_record = (
            participant_semantics[
                segment_name
            ]
        )


        print(
            f"\nSEGMENT: {segment_name}"
        )


        print(
            "\nCoarse summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "coarse_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\nFocused summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "focused_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


# ============================================================
# MAIN INSPECTION FUNCTION
# ============================================================

def inspect_case(
    family,
    sample_index,
    show_turns=True,
    show_temporal_features=True,
    show_semantics=True,
    show_full_json=False,
):
    assert family in cases_by_family


    family_cases = cases_by_family[
        family
    ]


    sample_index = int(
        sample_index
    )


    assert (
        0
        <= sample_index
        < len(
            family_cases
        )
    )


    case = family_cases[
        sample_index
    ]


    print("=" * 88)
    print("CONSOLIDATION CASE INSPECTION")
    print("=" * 88)

    print(
        "Family:",
        family,
    )

    print(
        "Family position:",
        f"{sample_index + 1}/{len(family_cases)}",
    )

    print(
        "case_id:",
        case.get(
            "case_id"
        ),
    )

    print(
        "source_group_id:",
        case.get(
            "source_group_id"
        ),
    )

    print(
        "case_variant:",
        case.get(
            "case_variant"
        ),
    )

    print(
        "gold_binary_label:",
        case.get(
            "gold_binary_label"
        ),
    )

    print(
        "gold_anomaly_type:",
        case.get(
            "gold_anomaly_type"
        ),
    )

    print(
        "A_source_conversation:",
        case.get(
            "A_source_conversation"
        ),
    )

    print(
        "B_source_conversation:",
        case.get(
            "B_source_conversation"
        ),
    )


    print(
        "\nPARTICIPANTS"
    )


    display(
        pd.DataFrame([
            participant_row(
                case,
                "participant_A",
            ),

            participant_row(
                case,
                "participant_B",
            ),
        ])
    )


    print(
        "\nSEMANTIC COVERAGE"
    )


    print(
        json.dumps(
            case.get(
                "semantic_summary_coverage"
            ),
            indent=2,
            ensure_ascii=False,
        )
    )


    if show_turns:

        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT A FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_A_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT B FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_B_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_temporal_features:

        print(
            "\n" + "=" * 88
        )

        print(
            "LOCAL TEMPORAL FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "local_temporal_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "GLOBAL SHIFT FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "global_shift_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_semantics:

        display_semantics(
            case,
            "participant_A",
        )


        display_semantics(
            case,
            "participant_B",
        )


    if show_full_json:

        print(
            "\n" + "=" * 88
        )

        print(
            "COMPLETE SAMPLE JSON"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case,
                indent=2,
                ensure_ascii=False,
            )
        )


    return case


# ============================================================
# WIDGETS
# ============================================================

family_labels = {
    "normal": "NORMAL",
    "wrong_partner": "WRONG_PARTNER",
    "lag": "LAG",
    "silent_partner": "SILENT_PARTNER",
}


family_dropdown = widgets.Dropdown(
    options=[
        (
            family_labels[
                family
            ],
            family,
        )

        for family in [
            "normal",
            "wrong_partner",
            "lag",
            "silent_partner",
        ]
    ],

    value="normal",

    description="Family:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="450px"
    ),
)


case_dropdown = widgets.Dropdown(
    description="Sample:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="900px"
    ),
)


show_turns_checkbox = widgets.Checkbox(
    value=True,
    description="Show filtered turns",
    indent=False,
)


show_temporal_checkbox = widgets.Checkbox(
    value=True,
    description="Show temporal features",
    indent=False,
)


show_semantics_checkbox = widgets.Checkbox(
    value=True,
    description="Show semantic summaries",
    indent=False,
)


show_full_json_checkbox = widgets.Checkbox(
    value=False,
    description="Show complete JSON",
    indent=False,
)


inspector_output = widgets.Output()


def refresh_case_options(
    *args,
):
    family = family_dropdown.value


    family_cases = cases_by_family[
        family
    ]


    case_dropdown.options = [
        (
            (
                f"{index + 1:03d}/100 | "
                f'{case.get("case_id")} | '
                f'{case.get("case_variant")}'
            ),

            index,
        )

        for index, case
        in enumerate(
            family_cases
        )
    ]


    case_dropdown.value = 0


def render_case(
    *args,
):
    if case_dropdown.value is None:
        return


    with inspector_output:

        clear_output(
            wait=True
        )


        inspect_case(
            family=(
                family_dropdown.value
            ),

            sample_index=(
                case_dropdown.value
            ),

            show_turns=(
                show_turns_checkbox.value
            ),

            show_temporal_features=(
                show_temporal_checkbox.value
            ),

            show_semantics=(
                show_semantics_checkbox.value
            ),

            show_full_json=(
                show_full_json_checkbox.value
            ),
        )


family_dropdown.observe(
    refresh_case_options,
    names="value",
)


family_dropdown.observe(
    render_case,
    names="value",
)


case_dropdown.observe(
    render_case,
    names="value",
)


show_turns_checkbox.observe(
    render_case,
    names="value",
)


show_temporal_checkbox.observe(
    render_case,
    names="value",
)


show_semantics_checkbox.observe(
    render_case,
    names="value",
)


show_full_json_checkbox.observe(
    render_case,
    names="value",
)


refresh_case_options()


controls = widgets.VBox([
    family_dropdown,
    case_dropdown,

    widgets.HBox([
        show_turns_checkbox,
        show_temporal_checkbox,
    ]),

    widgets.HBox([
        show_semantics_checkbox,
        show_full_json_checkbox,
    ]),
])


print("=" * 88)
print("INTERACTIVE INSPECTOR READY")
print("=" * 88)

for family in [
    "normal",
    "wrong_partner",
    "lag",
    "silent_partner",
]:

    print(
        family,
        len(
            cases_by_family[
                family
            ]
        ),
    )


display(
    controls,
    inspector_output,
)


render_case()


INTERACTIVE INSPECTOR READY
normal 100
wrong_partner 100
lag 100
silent_partner 100


Output()

## 7. Load Qwen2.5-Omni Thinker


In [ ]:
import torch
from transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

print("Loaded:", MODEL_ID)

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/233k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-Omni-7B


# Ηelper Functions For Qwen Calls

In [ ]:
def extract_json_from_text(text: str):
    text = text.strip()

    # remove markdown fences
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"```$", "", text.strip())

    try:
        return json.loads(text)
    except Exception:
        pass

    # find first balanced JSON object
    start = text.find("{")
    if start == -1:
        return {"parse_error": True, "raw_output": text}

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i+1]
                try:
                    return json.loads(candidate)
                except Exception:
                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }

    return {"parse_error": True, "raw_output": text}

def qwen_video_text(video_path: str, prompt: str, max_new_tokens: int = 350):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are Qwen, a virtual human developed by the Qwen Team, "
                        "Alibaba Group, capable of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    )
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "video": video_path, "fps": 7.0},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True
    )

    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    )

    inputs = inputs.to(model.device)

    print("Input tokens:", inputs["input_ids"].shape[1])

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated

def qwen_text_only(prompt: str, max_new_tokens: int = 700):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={
            "padding": True,
        },
    )

    inputs = {
        k: v.to(model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    input_token_count = int(
        inputs["input_ids"].shape[-1]
    )


    print(
        "Input tokens:",
        input_token_count,
    )


    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated


# Shared frozen NORMAL reference text

This is the same reference-text construction used by the original binary experiments. No anomaly-specific profile is added here; R2 and R3 load their exact saved prompts, which already contain the frozen LAG₂/LAG₃ profile text.

In [ ]:
# ============================================================
# BUILD AND PRINT THE SELECTED NORMAL-ONLY REFERENCE TEXT
#
# Only the frozen NORMAL profile is used.
#
# Included base metrics:
#   - num_offsets
#   - offset_mean
#   - offset_median
#   - offset_max
#   - offset_p75
#   - offset_p90
#   - percent_above_1_5
#   - clean_overlap_seconds
#
# Included global metrics:
#   - best_B_correction_shift_seconds__mean
#   - best_B_correction_shift_seconds__median
#   - estimated_B_lateness_seconds__mean
#   - estimated_B_lateness_seconds__median
#   - alignment_score_gain_vs_zero__mean
#   - alignment_score_gain_vs_zero__median
#   - best_num_bilateral_events__mean
#   - best_event_coverage__mean
#
# No other frozen statistics are exposed.
# ============================================================

import json
import math
import numbers


# ============================================================
# STRICT NORMAL PROFILE AUDIT
# ============================================================

assert isinstance(
    frozen_normal_base_statistics,
    dict,
)


assert isinstance(
    frozen_normal_shift_statistics,
    dict,
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# ============================================================
# NUMERIC VALIDATION HELPER
# ============================================================

def require_finite_numeric(
    record,
    field_name,
):
    """
    Read one required numeric field and verify
    that it exists and contains a finite number.
    """

    assert field_name in record, (
        f"Missing required field: {field_name}"
    )


    value = record[
        field_name
    ]


    assert isinstance(
        value,
        numbers.Real,
    ) and not isinstance(
        value,
        bool,
    ), (
        f"Expected numeric value for {field_name}, "
        f"found {type(value).__name__}: {value}"
    )


    value = float(
        value
    )


    assert math.isfinite(
        value
    ), (
        f"Non-finite value for {field_name}: {value}"
    )


    return value


def normalize_negative_zero(
    value,
):
    """
    Convert -0.0 to 0.0 for cleaner prompt text.
    """

    if abs(
        value
    ) < 1e-12:

        return 0.0


    return value


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL BASE STATISTICS
# ============================================================

normal_num_offsets = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "num_offsets",
    )
)


normal_offset_mean = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_mean",
    )
)


normal_offset_median = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_median",
    )
)


normal_offset_max = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_max",
    )
)


normal_offset_p75 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p75",
    )
)


normal_offset_p90 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p90",
    )
)


normal_percent_above_1_5 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "percent_above_1_5",
    )
)


normal_clean_overlap_seconds = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "clean_overlap_seconds",
    )
)


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL GLOBAL-SHIFT STATISTICS
#
# IMPORTANT:
# These fields use a double underscore before
# "mean" and "median".
# ============================================================

normal_best_shift_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__mean",
    )
)


normal_best_shift_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__median",
    )
)


normal_lateness_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__mean",
    )
)


normal_lateness_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__median",
    )
)


normal_alignment_gain_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__mean",
    )
)


normal_alignment_gain_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__median",
    )
)


normal_bilateral_events_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_num_bilateral_events__mean",
    )
)


normal_event_coverage_fraction = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_event_coverage__mean",
    )
)


# ============================================================
# CLEAN DISPLAY VALUES
# ============================================================

normal_best_shift_mean = (
    normalize_negative_zero(
        normal_best_shift_mean
    )
)


normal_best_shift_median = (
    normalize_negative_zero(
        normal_best_shift_median
    )
)


normal_lateness_mean = (
    normalize_negative_zero(
        normal_lateness_mean
    )
)


normal_lateness_median = (
    normalize_negative_zero(
        normal_lateness_median
    )
)


normal_event_coverage_percent = (
    100.0
    * normal_event_coverage_fraction
)


# ============================================================
# STORE THE EXACT VALUES USED IN THE PROMPT
# ============================================================

normal_base_reference_values = {
    "num_signed_offsets": (
        normal_num_offsets
    ),

    "mean_signed_offset_seconds": (
        normal_offset_mean
    ),

    "median_signed_offset_seconds": (
        normal_offset_median
    ),

    "maximum_signed_offset_seconds": (
        normal_offset_max
    ),

    "p75_signed_offset_seconds": (
        normal_offset_p75
    ),

    "p90_signed_offset_seconds": (
        normal_offset_p90
    ),

    "percent_offsets_above_1_5_seconds": (
        normal_percent_above_1_5
    ),

    "filtered_clean_overlap_seconds": (
        normal_clean_overlap_seconds
    ),
}


normal_global_reference_values = {
    "mean_best_B_correction_shift_seconds": (
        normal_best_shift_mean
    ),

    "median_best_B_correction_shift_seconds": (
        normal_best_shift_median
    ),

    "mean_estimated_B_lateness_seconds": (
        normal_lateness_mean
    ),

    "median_estimated_B_lateness_seconds": (
        normal_lateness_median
    ),

    "mean_alignment_score_gain_vs_zero": (
        normal_alignment_gain_mean
    ),

    "median_alignment_score_gain_vs_zero": (
        normal_alignment_gain_median
    ),

    "mean_num_bilateral_alignment_events": (
        normal_bilateral_events_mean
    ),

    "mean_bilateral_event_coverage_percent": (
        normal_event_coverage_percent
    ),
}


assert len(
    normal_base_reference_values
) == 8


assert len(
    normal_global_reference_values
) == 8


# ============================================================
# BUILD NORMAL LOCAL-TEMPORAL REFERENCE TEXT
# ============================================================

normal_base_reference_text = f"""
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around {normal_num_offsets:.2f}.
- Mean signed offset is around {normal_offset_mean:.2f} seconds.
- Median signed offset is around {normal_offset_median:.2f} seconds.
- Maximum signed offset is around {normal_offset_max:.2f} seconds.
- P75 signed offset is around {normal_offset_p75:.2f} seconds.
- P90 signed offset is around {normal_offset_p90:.2f} seconds.
- Approximately {normal_percent_above_1_5:.1f}% of offsets are above 1.5 seconds.
- Filtered clean overlap is around {normal_clean_overlap_seconds:.2f} seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.
""".strip()


# ============================================================
# BUILD NORMAL GLOBAL-ALIGNMENT REFERENCE TEXT
# ============================================================

normal_global_reference_text = f"""
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around {normal_best_shift_mean:.2f} seconds.
- Median best B correction shift is around {normal_best_shift_median:.2f} seconds.
- Mean estimated B lateness is around {normal_lateness_mean:.2f} seconds.
- Median estimated B lateness is around {normal_lateness_median:.2f} seconds.
- Mean alignment score gain versus zero shift is around {normal_alignment_gain_mean:.3f}.
- Median alignment score gain versus zero shift is around {normal_alignment_gain_median:.3f}.
- Mean number of bilateral alignment events is around {normal_bilateral_events_mean:.2f}.
- Mean bilateral event coverage is around {normal_event_coverage_percent:.2f}%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.
""".strip()


# ============================================================
# PRINT BOTH PROMPT SECTIONS
# ============================================================

print("=" * 88)
print("normal_base_reference_text")
print("=" * 88)

print(
    normal_base_reference_text
)


print("\n" + "=" * 88)
print("normal_global_reference_text")
print("=" * 88)

print(
    normal_global_reference_text
)


# ============================================================
# PRINT THE EXACT STRUCTURED VALUES USED
# ============================================================

print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL BASE VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_base_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL GLOBAL VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_global_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# STRICT CONTENT AUDIT
# ============================================================

combined_reference_text = (
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


# No anomaly-specific labels or profiles.
for forbidden_text in [
    "LAG +1",
    "LAG +2",
    "LAG +3",
    "wrong_partner",
    "silent_partner",
    "anomaly_type",
]:

    assert (
        forbidden_text.lower()
        not in combined_reference_text.lower()
    )


# Unwanted frozen reference fields must not appear.
for excluded_text in [
    "original A turns",
    "original B turns",
    "removed A backchannels",
    "removed B backchannels",
    "filtered A turns",
    "filtered B turns",
    "number of negative",
    "number of positive",
    "minimum signed offset",
    "clean overlap percentage",
]:

    assert (
        excluded_text.lower()
        not in combined_reference_text.lower()
    )


# All required selected metrics must appear.
for required_text in [
    "Number of signed offsets",
    "Mean signed offset",
    "Median signed offset",
    "Maximum signed offset",
    "P75 signed offset",
    "P90 signed offset",
    "offsets are above 1.5 seconds",
    "Filtered clean overlap",
    "Mean best B correction shift",
    "Median best B correction shift",
    "Mean estimated B lateness",
    "Median estimated B lateness",
    "Mean alignment score gain",
    "Median alignment score gain",
    "Mean number of bilateral alignment events",
    "Mean bilateral event coverage",
]:

    assert (
        required_text
        in combined_reference_text
    )


print("\n" + "=" * 88)
print("SELECTED NORMAL-ONLY REFERENCE TEXT READY")
print("=" * 88)

print(
    "Base reference metrics included:",
    len(
        normal_base_reference_values
    ),
)

print(
    "Global reference metrics included:",
    len(
        normal_global_reference_values
    ),
)

print(
    "No additional frozen statistics were exposed."
)

print(
    "No explicit anomaly profile or anomaly type was included."
)

normal_base_reference_text
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.
- Filtered clean overlap is around 6.25 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

normal_global_reference_text
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.

# Shared full-semantics input and reasoning helpers

The input projection below is copied from the original binary consolidation notebook. It passes exactly the same participation, turn, temporal, coarse-semantic, and focused-semantic fields.

The reasoning schema is categorical rather than free-form so that it can be parsed and inspected consistently.

In [ ]:

# ============================================================
# SHARED STRUCTURED-REASONING EXPERIMENT HELPERS
#
# The model-facing input projection is copied from the original
# binary consolidation notebook:
#   - same participation fields
#   - same filtered turns
#   - same local temporal fields
#   - same global temporal fields
#   - same coarse summaries
#   - same focused summaries
#
# The three source prompts are loaded from their exact saved
# prompt_template.txt files.
#
# The only prompt change is replacement of the original binary
# OUTPUT block with one common structured-reasoning JSON schema.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import copy
import hashlib
import json
import re
import time

import pandas as pd
import torch

from tqdm.auto import tqdm


import difflib

MAX_NEW_TOKENS_REASONING = 256
MAX_NEW_TOKENS = MAX_NEW_TOKENS_REASONING

LABELS = [
    "NORMAL",
    "ANOMALOUS",
]


# ============================================================
# EXACT SEMANTIC FIELDS PASSED TO THE MODEL
# ============================================================

COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEGMENT_MAP = {
    "segment_0_0_to_60_seconds": (
        "segment_0"
    ),

    "segment_1_60_to_120_seconds": (
        "segment_1"
    ),
}


# ============================================================
# EXACT TEMPORAL FIELDS PASSED TO THE MODEL
# ============================================================

LOCAL_FEATURE_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


GLOBAL_FEATURE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# FIELDS THAT MUST NEVER APPEAR IN THE MODEL PROMPT
# ============================================================

FORBIDDEN_PROMPT_KEYS = [
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "conversation_id",
    "participant_id",
    "metadata_path",

    "num_raw_vad_entries",

    "A_source_conversation",
    "B_source_conversation",

    "semantic_summary_source",
    "semantic_summary_coverage",
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def canonical_json(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def sha256_text(
    text,
):
    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def select_exact_fields(
    record,
    fields,
    context,
):
    assert isinstance(
        record,
        dict,
    ), (
        f"Expected dictionary for {context}"
    )


    missing_fields = [
        field

        for field in fields

        if field not in record
    ]


    assert not missing_fields, (
        f"Missing fields in {context}: "
        f"{missing_fields}"
    )


    return {
        field: copy.deepcopy(
            record[
                field
            ]
        )

        for field in fields
    }


# ============================================================
# TURN PROJECTION
#
# Database format:
#   {"start": 1.2, "end": 3.4}
#
# Prompt format:
#   [1.2, 3.4]
# ============================================================

def compact_turns(
    turns,
    context,
):
    assert isinstance(
        turns,
        list,
    ), (
        f"Expected list for {context}"
    )


    compact = []


    for index, turn in enumerate(
        turns
    ):

        assert isinstance(
            turn,
            dict,
        ), (
            f"Invalid turn in {context} "
            f"at index {index}"
        )


        assert "start" in turn
        assert "end" in turn


        start = float(
            turn[
                "start"
            ]
        )


        end = float(
            turn[
                "end"
            ]
        )


        assert end >= start


        compact.append([
            start,
            end,
        ])


    return compact


# ============================================================
# SEMANTIC INPUT PROJECTION
#
# Participant IDs and other metadata are deliberately excluded.
# ============================================================

def build_semantic_input(
    case,
):
    semantic_input = {}


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        role_output = {}


        for (
            database_segment_name,
            prompt_segment_name,
        ) in SEGMENT_MAP.items():

            segment_record = (
                participant_semantics[
                    database_segment_name
                ]
            )


            coarse_summary = (
                select_exact_fields(
                    segment_record[
                        "coarse_summary"
                    ],

                    COARSE_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "coarse_summary"
                    ),
                )
            )


            focused_summary = (
                select_exact_fields(
                    segment_record[
                        "focused_summary"
                    ],

                    FOCUSED_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "focused_summary"
                    ),
                )
            )


            # Legacy focused-summary speaks must not exist.
            assert (
                "speaks"
                not in focused_summary
            )


            role_output[
                prompt_segment_name
            ] = {
                "coarse_summary": (
                    coarse_summary
                ),

                "focused_summary": (
                    focused_summary
                ),
            }


        semantic_input[
            role
        ] = role_output


    return semantic_input


# ============================================================
# BUILD THE EXACT MODEL INPUT
# ============================================================

def build_binary_model_input(
    case,
):
    local_features = (
        select_exact_fields(
            case[
                "local_temporal_features"
            ],

            LOCAL_FEATURE_FIELDS,

            "local_temporal_features",
        )
    )


    global_features_raw = (
        select_exact_fields(
            case[
                "global_shift_features"
            ],

            GLOBAL_FEATURE_FIELDS,

            "global_shift_features",
        )
    )


    # Stored in the database as a fraction.
    # Presented in the prompt as a percentage so that it is
    # directly comparable with the frozen NORMAL percentage.
    event_coverage_fraction = float(
        global_features_raw.pop(
            "best_event_coverage"
        )
    )


    global_features = {
        **global_features_raw,

        "best_event_coverage_percent": round(
            100.0
            * event_coverage_fraction,

            6,
        ),
    }


    payload = {
        "analysis_duration_seconds": float(
            case[
                "analysis_duration_seconds"
            ]
        ),

        "participant_A": {
            "speaks": bool(
                case[
                    "participant_A"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_A_filtered_turns"
                ],

                "participant_A_filtered_turns",
            ),
        },

        "participant_B": {
            "speaks": bool(
                case[
                    "participant_B"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_B_filtered_turns"
                ],

                "participant_B_filtered_turns",
            ),
        },

        "local_temporal_features": (
            local_features
        ),

        "global_shift_features": (
            global_features
        ),

        "semantic_summaries": (
            build_semantic_input(
                case
            )
        ),
    }


    assert set(
        payload
    ) == {
        "analysis_duration_seconds",
        "participant_A",
        "participant_B",
        "local_temporal_features",
        "global_shift_features",
        "semantic_summaries",
    }


    return payload


# ============================================================
# BUILD THE FINAL PROMPT FOR ONE CASE
# ============================================================

def build_binary_prompt_from_template(
    case,
    prompt_template,
):
    payload = build_binary_model_input(
        case
    )


    prompt = (
        prompt_template
        .format(
            normal_base_reference_text=(
                normal_base_reference_text
            ),

            normal_global_reference_text=(
                normal_global_reference_text
            ),

            duration_seconds=(
                payload[
                    "analysis_duration_seconds"
                ]
            ),

            participant_A_speaks=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_A_turns=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            participant_B_speaks=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_B_turns=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            local_temporal_features=(
                json.dumps(
                    payload[
                        "local_temporal_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            global_shift_features=(
                json.dumps(
                    payload[
                        "global_shift_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            semantic_summaries=(
                json.dumps(
                    payload[
                        "semantic_summaries"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),
        )
    )


    # Strict leakage audit.
    prompt_lower = prompt.lower()


    for forbidden_key in (
        FORBIDDEN_PROMPT_KEYS
    ):

        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )


    return (
        prompt,
        payload,
    )


# ============================================================
# TEXT-ONLY QWEN INFERENCE
#
# The existing qwen_text_only helper used 700 output tokens.
# Here only 64 are needed because the expected output is:
#
#   {"label": "NORMAL"}
#
# or:
#
#   {"label": "ANOMALOUS"}
# ============================================================

def qwen_text_only_binary(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]


    inputs = (
        processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",

            processor_kwargs={
                "padding": True,
            },
        )
    )


    inputs = {
        key: (
            value.to(
                model.device
            )
            if hasattr(
                value,
                "to",
            )
            else value
        )

        for key, value
        in inputs.items()
    }


    input_token_count = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            use_cache=True,
        )


    generated_ids = output_ids[
        :,
        inputs[
            "input_ids"
        ].shape[1]:,
    ]


    raw_output = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
    )


    return (
        raw_output,
        input_token_count,
    )


# ============================================================
# ROBUST JSON PARSING
# ============================================================

def extract_first_json_object(
    text,
):
    cleaned = text.strip()


    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )


    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )


    try:

        return json.loads(
            cleaned
        )

    except Exception:

        pass


    decoder = json.JSONDecoder()


    possible_starts = [
        index

        for index, character
        in enumerate(
            cleaned
        )

        if character == "{"
    ]


    for start in possible_starts:

        try:

            parsed, _ = (
                decoder.raw_decode(
                    cleaned[
                        start:
                    ]
                )
            )


            return parsed

        except Exception:

            continue


    return None

# ============================================================
# STRUCTURED REASONING SCHEMA
# ============================================================

REASONING_SCHEMA_KEYS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
    "label",
]


REASONING_ALLOWED_VALUES = {
    "participation_assessment": {
        "VALID",
        "INVALID",
    },

    "local_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "global_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "semantic_assessment": {
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    },

    "decisive_dimension": {
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    },

    "label": {
        "NORMAL",
        "ANOMALOUS",
    },
}


OLD_BINARY_OUTPUT_BLOCK = """
============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include reasoning.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


STRUCTURED_REASONING_OUTPUT_BLOCK = """
============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


def build_structured_reasoning_prompt_template(
    source_prompt_template,
):
    assert isinstance(
        source_prompt_template,
        str,
    )


    assert source_prompt_template.endswith(
        OLD_BINARY_OUTPUT_BLOCK
    ), (
        "The saved source prompt does not end with the exact "
        "original binary OUTPUT block."
    )


    reasoning_prompt_template = (
        source_prompt_template[
            :-len(
                OLD_BINARY_OUTPUT_BLOCK
            )
        ]
        + STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    reconstructed_source_prompt = (
        reasoning_prompt_template[
            :-len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            )
        ]
        + OLD_BINARY_OUTPUT_BLOCK
    )


    assert (
        reconstructed_source_prompt
        == source_prompt_template
    ), (
        "Unexpected source-prompt modification."
    )


    return reasoning_prompt_template


def parse_structured_reasoning_prediction(
    raw_output,
):
    parsed = extract_first_json_object(
        raw_output
    )


    normalized = {
        key: None
        for key in REASONING_SCHEMA_KEYS
    }


    schema_errors = []


    if isinstance(
        parsed,
        dict,
    ):

        for key in REASONING_SCHEMA_KEYS:

            if key in parsed:

                normalized[
                    key
                ] = str(
                    parsed[
                        key
                    ]
                ).strip().upper()


        for key in REASONING_SCHEMA_KEYS:

            if normalized[
                key
            ] not in REASONING_ALLOWED_VALUES[
                key
            ]:

                schema_errors.append(
                    (
                        f"{key}: "
                        f"{normalized[key]}"
                    )
                )


        exact_keys = (
            set(
                parsed.keys()
            )
            == set(
                REASONING_SCHEMA_KEYS
            )
        )


        schema_exact = (
            exact_keys
            and
            not schema_errors
        )


        label = normalized[
            "label"
        ]


        if label in LABELS:

            return {
                "prediction": label,

                "parse_mode": (
                    "structured_json"
                ),

                "schema_exact": bool(
                    schema_exact
                ),

                "parsed_output": parsed,

                "schema_errors": (
                    schema_errors
                ),

                **{
                    key: normalized[
                        key
                    ]

                    for key in (
                        REASONING_SCHEMA_KEYS
                    )

                    if key != "label"
                },
            }


    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )


    if plain_output in LABELS:

        return {
            "prediction": plain_output,

            "parse_mode": (
                "exact_plaintext_fallback"
            ),

            "schema_exact": False,

            "parsed_output": None,

            "schema_errors": [
                "Structured reasoning fields missing."
            ],

            **{
                key: None

                for key in (
                    REASONING_SCHEMA_KEYS
                )

                if key != "label"
            },
        }


    return {
        "prediction": None,

        "parse_mode": "invalid",

        "schema_exact": False,

        "parsed_output": parsed,

        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid structured JSON prediction."
            ]
        ),

        **{
            key: normalized[
                key
            ]

            for key in (
                REASONING_SCHEMA_KEYS
            )

            if key != "label"
        },
    }


# ============================================================
# FULL-SEMANTICS PAYLOAD AUDIT
# ============================================================

def collect_nested_keys_reasoning(
    value,
):
    keys = set()


    if isinstance(
        value,
        dict,
    ):

        for key, nested_value in value.items():

            keys.add(
                str(key)
            )


            keys.update(
                collect_nested_keys_reasoning(
                    nested_value
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            keys.update(
                collect_nested_keys_reasoning(
                    item
                )
            )


    return keys


FULL_SEMANTIC_REQUIRED_KEYS = [
    "semantic_summaries",
    "coarse_summary",
    "focused_summary",

    "speech_content_summary",
    "apparent_topic",

    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


# ============================================================
# EXACT PROMPT CONFIGURATION
# ============================================================

def prepare_reasoning_experiment(
    *,
    experiment_version,
    source_experiment_name,
    experiment_title,
    required_source_markers,
    forbidden_source_markers,
    temporal_profiles_used,
    assessment_policy,
):
    source_experiment_dir = (
        OUT_DIR
        / source_experiment_name
    )


    source_prompt_path = (
        source_experiment_dir
        / "prompt_template.txt"
    )


    assert source_prompt_path.exists(), (
        "Saved source prompt not found:\n"
        f"{source_prompt_path}"
    )


    source_prompt_template = (
        source_prompt_path.read_text(
            encoding="utf-8"
        )
    )


    for marker in required_source_markers:

        assert marker in source_prompt_template, (
            "Required source-prompt marker missing: "
            f"{marker}"
        )


    for marker in forbidden_source_markers:

        assert marker not in source_prompt_template, (
            "Forbidden source-prompt marker found: "
            f"{marker}"
        )


    for semantic_marker in [
        "Coarse semantic information:",
        "Focused semantic information:",
        "- detailed_speech_summary",
        "- main_topic",
        "- secondary_topics",
        "- key_semantic_details",
    ]:

        assert semantic_marker in source_prompt_template, (
            "The source prompt is not a full-semantics prompt. "
            f"Missing: {semantic_marker}"
        )


    reasoning_prompt_template = (
        build_structured_reasoning_prompt_template(
            source_prompt_template
        )
    )


    experiment_dir = (
        OUT_DIR
        / experiment_version
    )


    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),

        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),

        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),

        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),

        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),

        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),

        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),

        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),

        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),

        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),

        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),

        "source_prompt_copy": (
            experiment_dir
            / "source_prompt_template.txt"
        ),

        "prompt_diff": (
            experiment_dir
            / "prompt_diff_vs_source.txt"
        ),

        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }


    source_prompt_sha256 = sha256_text(
        source_prompt_template
    )


    reasoning_prompt_sha256 = sha256_text(
        reasoning_prompt_template
    )


    reasoning_schema_sha256 = sha256_text(
        STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )


    paths[
        "prompt_template"
    ].write_text(
        reasoning_prompt_template,
        encoding="utf-8",
    )


    paths[
        "source_prompt_copy"
    ].write_text(
        source_prompt_template,
        encoding="utf-8",
    )


    prompt_diff_lines = difflib.unified_diff(
        source_prompt_template.splitlines(),
        reasoning_prompt_template.splitlines(),

        fromfile=(
            f"{source_experiment_name}/prompt_template.txt"
        ),

        tofile=(
            f"{experiment_version}/prompt_template.txt"
        ),

        lineterm="",
    )


    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(
            prompt_diff_lines
        ),
        encoding="utf-8",
    )


    example_prompt, example_payload = (
        build_binary_prompt_from_template(
            consolidation_cases[0],
            reasoning_prompt_template,
        )
    )


    payload_keys = collect_nested_keys_reasoning(
        example_payload
    )


    for required_key in FULL_SEMANTIC_REQUIRED_KEYS:

        assert required_key in payload_keys, (
            "Full-semantic input field missing: "
            f"{required_key}"
        )


    assert (
        "STRUCTURED REASONING OUTPUT"
        in example_prompt
    )


    assert (
        "Do not include reasoning."
        not in reasoning_prompt_template[
            -len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            ):
        ]
    )


    manifest = {
        "experiment_version": (
            experiment_version
        ),

        "experiment_title": (
            experiment_title
        ),

        "source_experiment": (
            source_experiment_name
        ),

        "source_prompt_path": str(
            source_prompt_path
        ),

        "source_prompt_sha256": (
            source_prompt_sha256
        ),

        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),

        "reasoning_schema_sha256": (
            reasoning_schema_sha256
        ),

        "normal_reference_sha256": (
            normal_reference_sha256
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            temporal_profiles_used
        ),

        "assessment_policy": (
            assessment_policy
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "model_id": (
            MODEL_ID
        ),

        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },

        "only_prompt_change": (
            "The exact original binary OUTPUT block was "
            "replaced by the common structured-reasoning "
            "OUTPUT block."
        ),

        "all_model_facing_evidence_unchanged": (
            True
        ),

        "all_pre_output_prompt_text_unchanged": (
            True
        ),
    }


    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    config = {
        **manifest,

        "experiment_dir": (
            experiment_dir
        ),

        "source_prompt_template": (
            source_prompt_template
        ),

        "reasoning_prompt_template": (
            reasoning_prompt_template
        ),

        "paths": paths,
    }


    print("=" * 88)
    print(
        f"{experiment_title} — CONFIGURATION READY"
    )
    print("=" * 88)

    print(
        "Experiment version:",
        experiment_version,
    )

    print(
        "Source experiment:",
        source_experiment_name,
    )

    print(
        "Source prompt SHA256:",
        source_prompt_sha256,
    )

    print(
        "Reasoning prompt SHA256:",
        reasoning_prompt_sha256,
    )

    print(
        "Semantic input:",
        "coarse_and_focused",
    )

    print(
        "Focused summaries used:",
        True,
    )

    print(
        "Temporal profiles:",
        temporal_profiles_used,
    )

    print(
        "Assessment policy:",
        assessment_policy,
    )

    print(
        "Only source-prompt change:",
        (
            "binary OUTPUT block -> "
            "structured reasoning OUTPUT block"
        ),
    )

    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )

    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )


    return config


# ============================================================
# CHECKPOINTED INFERENCE
# ============================================================

def reasoning_utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def reasoning_atomic_write_json(
    path,
    data,
):
    path = Path(
        path
    )


    temporary_path = path.with_suffix(
        path.suffix
        + ".tmp"
    )


    temporary_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    temporary_path.replace(
        path
    )


def create_reasoning_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "model_id": (
            MODEL_ID
        ),

        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),

        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),

        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),

        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "created_at_utc": (
            reasoning_utc_now()
        ),

        "updated_at_utc": (
            reasoning_utc_now()
        ),

        "records": {},
    }


def run_reasoning_experiment(
    config,
):
    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )


    if prediction_cache_path.exists():

        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )


        for key in [
            "experiment_version",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "temporal_profiles_used",
            "assessment_policy",
            "max_new_tokens",
        ]:

            expected_value = (
                create_reasoning_cache(
                    config
                )[
                    key
                ]
            )


            assert (
                prediction_cache[
                    key
                ]
                == expected_value
            ), (
                f"Cache mismatch for {key}"
            )


        print(
            "Resuming cache:",
            prediction_cache_path,
        )

        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )


    else:

        prediction_cache = (
            create_reasoning_cache(
                config
            )
        )


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


        print(
            "Created cache:",
            prediction_cache_path,
        )


    ordered_cases = sorted(
        consolidation_cases,

        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )


    assert len(
        ordered_cases
    ) == 400


    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):

        case_id = str(
            case[
                "case_id"
            ]
        )


        prompt, model_input_payload = (
            build_binary_prompt_from_template(
                case,
                config[
                    "reasoning_prompt_template"
                ],
            )
        )


        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )


        for required_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):

            assert required_key in (
                current_payload_keys
            ), (
                f"Missing full-semantic field "
                f"for case {case_id}: "
                f"{required_key}"
            )


        prompt_sha256 = sha256_text(
            prompt
        )


        input_payload_sha256 = sha256_text(
            canonical_json(
                model_input_payload
            )
        )


        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        if (
            existing_record is not None
            and
            existing_record.get(
                "prediction"
            ) in LABELS
        ):

            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )


            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )


            continue


        started = time.perf_counter()


        try:

            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )


            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )


            generation_error = None


        except Exception as exc:

            raw_output = ""


            input_token_count = None


            parsed_result = {
                "prediction": None,
                "parse_mode": "generation_error",
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }


            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )


            if torch.cuda.is_available():

                torch.cuda.empty_cache()


        elapsed_seconds = (
            time.perf_counter()
            - started
        )


        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prompt_sha256": (
                prompt_sha256
            ),

            "input_payload_sha256": (
                input_payload_sha256
            ),

            "semantic_input": (
                "coarse_and_focused"
            ),

            "focused_summaries_used": True,

            "input_token_count": (
                input_token_count
            ),

            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),

            "raw_output": (
                raw_output
            ),

            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),

            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),

            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),

            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),

            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),

            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),

            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),

            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),

            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),

            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),

            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),

            "generation_error": (
                generation_error
            ),

            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),

            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }


        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)

    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )

    print(
        "Prediction cache:",
        prediction_cache_path,
    )


    return prediction_cache


# ============================================================
# EVALUATION
# ============================================================

def summarize_reasoning_group(
    group,
):
    valid_group = group[
        group[
            "valid_prediction"
        ]
    ]


    return {
        "total_cases": int(
            len(
                group
            )
        ),

        "valid_predictions": int(
            len(
                valid_group
            )
        ),

        "invalid_predictions": int(
            len(
                group
            )
            -
            len(
                valid_group
            )
        ),

        "predicted_NORMAL": int(
            (
                valid_group[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),

        "predicted_ANOMALOUS": int(
            (
                valid_group[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "correct_predictions": int(
            group[
                "correct"
            ].sum()
        ),

        "accuracy_on_valid": (
            float(
                (
                    valid_group[
                        "gold_label"
                    ]
                    ==
                    valid_group[
                        "prediction"
                    ]
                ).mean()
            )
            if len(
                valid_group
            )
            else float(
                "nan"
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            group[
                "correct"
            ].mean()
        ),

        "exact_schema_rate": float(
            group[
                "schema_exact"
            ].mean()
        ),
    }


def evaluate_reasoning_experiment(
    config,
):
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        matthews_corrcoef,
        precision_score,
        recall_score,
    )

    from IPython.display import display


    prediction_cache = json.loads(
        config[
            "paths"
        ][
            "prediction_cache"
        ].read_text(
            encoding="utf-8"
        )
    )


    assert (
        prediction_cache[
            "experiment_version"
        ]
        == config[
            "experiment_version"
        ]
    )


    assert (
        prediction_cache[
            "reasoning_prompt_sha256"
        ]
        == config[
            "reasoning_prompt_sha256"
        ]
    )


    result_rows = []


    for case in consolidation_cases:

        case_id = str(
            case[
                "case_id"
            ]
        )


        record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        result_rows.append({
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prediction": (
                None
                if record is None
                else record.get(
                    "prediction"
                )
            ),

            "participation_assessment": (
                None
                if record is None
                else record.get(
                    "participation_assessment"
                )
            ),

            "local_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "local_temporal_assessment"
                )
            ),

            "global_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "global_temporal_assessment"
                )
            ),

            "temporal_assessment": (
                None
                if record is None
                else record.get(
                    "temporal_assessment"
                )
            ),

            "semantic_assessment": (
                None
                if record is None
                else record.get(
                    "semantic_assessment"
                )
            ),

            "decisive_dimension": (
                None
                if record is None
                else record.get(
                    "decisive_dimension"
                )
            ),

            "parse_mode": (
                "missing"
                if record is None
                else record.get(
                    "parse_mode"
                )
            ),

            "schema_exact": (
                False
                if record is None
                else bool(
                    record.get(
                        "schema_exact",
                        False,
                    )
                )
            ),

            "schema_errors": (
                None
                if record is None
                else json.dumps(
                    record.get(
                        "schema_errors",
                        [],
                    ),
                    ensure_ascii=False,
                )
            ),

            "input_token_count": (
                None
                if record is None
                else record.get(
                    "input_token_count"
                )
            ),

            "elapsed_seconds": (
                None
                if record is None
                else record.get(
                    "elapsed_seconds"
                )
            ),

            "raw_output": (
                ""
                if record is None
                else record.get(
                    "raw_output",
                    "",
                )
            ),

            "generation_error": (
                None
                if record is None
                else record.get(
                    "generation_error"
                )
            ),
        })


    results_df = pd.DataFrame(
        result_rows
    )


    results_df[
        "valid_prediction"
    ] = results_df[
        "prediction"
    ].isin(
        LABELS
    )


    results_df[
        "correct"
    ] = (
        results_df[
            "valid_prediction"
        ]
        &
        (
            results_df[
                "gold_label"
            ]
            ==
            results_df[
                "prediction"
            ]
        )
    )


    results_df[
        "normal_with_invalid_participation"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "participation_assessment"
            ]
            == "INVALID"
        )
    )


    results_df[
        "normal_with_anomalous_temporal"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "temporal_assessment"
            ]
            == "ANOMALOUS"
        )
    )


    results_df[
        "normal_with_incompatible_semantics"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "semantic_assessment"
            ]
            == "INCOMPATIBLE"
        )
    )


    results_df[
        "reasoning_inconsistency"
    ] = (
        results_df[
            [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
            ]
        ].any(
            axis=1
        )
    )


    assert len(
        results_df
    ) == 400


    assert (
        results_df[
            "case_id"
        ].is_unique
    )


    valid_df = results_df[
        results_df[
            "valid_prediction"
        ]
    ].copy()


    invalid_df = results_df[
        ~results_df[
            "valid_prediction"
        ]
    ].copy()


    assert len(
        valid_df
    ) > 0


    y_true = valid_df[
        "gold_label"
    ]


    y_pred = valid_df[
        "prediction"
    ]


    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            "NORMAL",
            "ANOMALOUS",
        ],
    )


    confusion_df = pd.DataFrame(
        confusion,

        index=[
            "Gold NORMAL",
            "Gold ANOMALOUS",
        ],

        columns=[
            "Pred NORMAL",
            "Pred ANOMALOUS",
        ],
    )


    true_normal = int(
        confusion[
            0,
            0,
        ]
    )


    false_anomalous = int(
        confusion[
            0,
            1,
        ]
    )


    normal_recall = (
        true_normal
        /
        (
            true_normal
            + false_anomalous
        )
        if (
            true_normal
            + false_anomalous
        )
        else float(
            "nan"
        )
    )


    metrics = {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "total_cases": int(
            len(
                results_df
            )
        ),

        "valid_predictions": int(
            len(
                valid_df
            )
        ),

        "invalid_predictions": int(
            len(
                invalid_df
            )
        ),

        "exact_json_schema_rate": float(
            results_df[
                "schema_exact"
            ].mean()
        ),

        "reasoning_inconsistency_count": int(
            results_df[
                "reasoning_inconsistency"
            ].sum()
        ),

        "accuracy_valid_predictions": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            results_df[
                "correct"
            ].mean()
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "anomalous_precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_recall": float(
            recall_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_f1": float(
            f1_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "normal_recall_specificity": float(
            normal_recall
        ),

        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                y_true,
                y_pred,
            )
        ),

        "confusion_matrix_label_order": [
            "NORMAL",
            "ANOMALOUS",
        ],

        "confusion_matrix": (
            confusion.tolist()
        ),
    }


    family_rows = []


    for (
        family_name,
        family_group,
    ) in results_df.groupby(
        "case_family",
        sort=True,
    ):

        family_rows.append({
            "case_family": (
                family_name
            ),

            **summarize_reasoning_group(
                family_group
            ),
        })


    family_metrics_df = pd.DataFrame(
        family_rows
    )


    variant_rows = []


    for (
        variant_name,
        variant_group,
    ) in results_df.groupby(
        "case_variant",
        sort=True,
    ):

        variant_rows.append({
            "case_variant": (
                variant_name
            ),

            **summarize_reasoning_group(
                variant_group
            ),
        })


    variant_metrics_df = pd.DataFrame(
        variant_rows
    )


    classification_report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=[
                "NORMAL",
                "ANOMALOUS",
            ],
            output_dict=True,
            zero_division=0,
        )
    ).T


    error_df = results_df[
        (
            ~results_df[
                "valid_prediction"
            ]
        )
        |
        (
            results_df[
                "gold_label"
            ] != results_df[
                "prediction"
            ]
        )
    ].copy()


    reasoning_inconsistency_df = (
        results_df[
            results_df[
                "reasoning_inconsistency"
            ]
        ].copy()
    )


    assessment_rows = []


    for assessment_field in [
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
    ]:

        counts = (
            results_df[
                assessment_field
            ]
            .fillna(
                "MISSING"
            )
            .value_counts(
                dropna=False
            )
        )


        for value, count in counts.items():

            assessment_rows.append({
                "assessment_field": (
                    assessment_field
                ),

                "assessment_value": (
                    value
                ),

                "count": int(
                    count
                ),
            })


    assessment_distributions_df = (
        pd.DataFrame(
            assessment_rows
        )
    )


    source_group_rows = []


    for (
        source_group_id,
        source_group,
    ) in results_df.groupby(
        "source_group_id"
    ):

        source_group_rows.append({
            "source_group_id": (
                source_group_id
            ),

            "num_cases": int(
                len(
                    source_group
                )
            ),

            "all_predictions_valid": bool(
                source_group[
                    "valid_prediction"
                ].all()
            ),

            "all_cases_correct": bool(
                source_group[
                    "valid_prediction"
                ].all()
                and
                source_group[
                    "correct"
                ].all()
            ),
        })


    source_group_df = pd.DataFrame(
        source_group_rows
    )


    assert len(
        source_group_df
    ) == 100


    assert set(
        source_group_df[
            "num_cases"
        ]
    ) == {
        4
    }


    metrics[
        "source_group_exact_match_rate"
    ] = float(
        source_group_df[
            "all_cases_correct"
        ].mean()
    )


    paths = config[
        "paths"
    ]


    results_df.to_csv(
        paths[
            "predictions_csv"
        ],
        index=False,
    )


    reasoning_columns = [
        "case_id",
        "source_group_id",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "correct",
    ]


    results_df[
        reasoning_columns
    ].to_csv(
        paths[
            "reasoning_assessments_csv"
        ],
        index=False,
    )


    confusion_df.to_csv(
        paths[
            "confusion_matrix_csv"
        ]
    )


    family_metrics_df.to_csv(
        paths[
            "family_metrics_csv"
        ],
        index=False,
    )


    variant_metrics_df.to_csv(
        paths[
            "variant_metrics_csv"
        ],
        index=False,
    )


    error_df.to_csv(
        paths[
            "errors_csv"
        ],
        index=False,
    )


    reasoning_inconsistency_df.to_csv(
        paths[
            "reasoning_inconsistencies_csv"
        ],
        index=False,
    )


    assessment_distributions_df.to_csv(
        paths[
            "assessment_distributions_csv"
        ],
        index=False,
    )


    paths[
        "metrics_json"
    ].write_text(
        json.dumps(
            metrics,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    print("=" * 88)
    print(
        f"{config['experiment_title']} — RESULTS"
    )
    print("=" * 88)

    print(
        "Total cases:",
        metrics[
            "total_cases"
        ],
    )

    print(
        "Valid predictions:",
        metrics[
            "valid_predictions"
        ],
    )

    print(
        "Invalid predictions:",
        metrics[
            "invalid_predictions"
        ],
    )

    print(
        "Exact structured-schema rate:",
        (
            f'{metrics["exact_json_schema_rate"]:.4f}'
        ),
    )

    print(
        "Accuracy:",
        (
            f'{metrics["accuracy_valid_predictions"]:.4f}'
        ),
    )

    print(
        "Balanced accuracy:",
        (
            f'{metrics["balanced_accuracy"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS precision:",
        (
            f'{metrics["anomalous_precision"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS recall:",
        (
            f'{metrics["anomalous_recall"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS F1:",
        (
            f'{metrics["anomalous_f1"]:.4f}'
        ),
    )

    print(
        "NORMAL recall / specificity:",
        (
            f'{metrics["normal_recall_specificity"]:.4f}'
        ),
    )

    print(
        "MCC:",
        (
            f'{metrics["matthews_correlation_coefficient"]:.4f}'
        ),
    )

    print(
        "Matched source-group exact rate:",
        (
            f'{metrics["source_group_exact_match_rate"]:.4f}'
        ),
    )

    print(
        "Reasoning inconsistencies:",
        metrics[
            "reasoning_inconsistency_count"
        ],
    )


    print("\nCONFUSION MATRIX")

    display(
        confusion_df
    )


    print("\nCLASSIFICATION REPORT")

    display(
        classification_report_df
    )


    print("\nPER CASE FAMILY")

    display(
        family_metrics_df
    )


    print("\nPER EXACT CASE VARIANT")

    display(
        variant_metrics_df
    )


    print("\nASSESSMENT DISTRIBUTIONS")

    display(
        assessment_distributions_df
    )


    print(
        "\nSaved cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Saved prompt:",
        paths[
            "prompt_template"
        ],
    )

    print(
        "Saved prompt diff:",
        paths[
            "prompt_diff"
        ],
    )

    print(
        "Saved predictions:",
        paths[
            "predictions_csv"
        ],
    )

    print(
        "Saved reasoning assessments:",
        paths[
            "reasoning_assessments_csv"
        ],
    )

    print(
        "Saved errors:",
        paths[
            "errors_csv"
        ],
    )


    if len(
        invalid_df
    ):

        print(
            "\nWARNING: Invalid predictions were excluded "
            "from the confusion matrix."
        )


        display(
            invalid_df[
                [
                    "case_id",
                    "case_family",
                    "case_variant",
                    "parse_mode",
                    "raw_output",
                    "generation_error",
                ]
            ]
        )


    else:

        print(
            "\nAll 400 cases produced valid "
            "binary predictions."
        )


    return {
        "metrics": metrics,
        "results_df": results_df,
        "confusion_df": confusion_df,
        "family_metrics_df": family_metrics_df,
        "variant_metrics_df": variant_metrics_df,
        "error_df": error_df,
        "reasoning_inconsistency_df": (
            reasoning_inconsistency_df
        ),
        "assessment_distributions_df": (
            assessment_distributions_df
        ),
    }


# Phase A — Participation Branch Ablations

The exact saved binary source prompt for R1 is loaded from:

`binary_only_consolidation_normal_definition_v2/prompt_template.txt`

The full Structured R1 prompt is reconstructed using the same structured-reasoning output replacement as the original notebook. Each ablation then removes both:

1. the requested model-facing evidence fields, and
2. only the prompt statements that depend on those unavailable fields.

For **A1**, the complete participation branch is removed from the operational definition, dedicated participation section, final decision policy, structured output, and decisive-dimension options.

For **A1-S**, participation assessment remains but is based only on retained filtered turns.

For **A1-T**, participation assessment remains but is based only on retained `speaks` fields; raw-turn-specific instructions are removed.

All temporal and semantic branches, frozen NORMAL references, model settings, deterministic decoding, case order, checkpointing, and evaluation remain unchanged.


In [ ]:

# ============================================================
# PARTICIPATION-BRANCH ABLATION HELPERS
#
# This layer starts from the exact saved Structured R1 prompt:
#   binary_only_consolidation_normal_definition_v2
#   + the exact common Structured R1 output block.
#
# The model-facing input removals are controlled by:
#   include_speaks
#   include_filtered_turns
#
# The prompt is adapted consistently for every ablation:
#   - unavailable evidence is removed from AVAILABLE EVIDENCE,
#   - unavailable evidence instructions are removed or minimally
#     rewritten inside the relevant R1 sections,
#   - empty CURRENT CASE participant blocks are removed,
#   - the complete participation branch and its output field are
#     removed only for A1.
#
# Local temporal, global temporal, coarse semantics, focused
# semantics, NORMAL references, model, decoding, case order,
# checkpointing, and evaluation remain unchanged.
# ============================================================

from IPython.display import display

R1_SOURCE_EXPERIMENT_NAME = (
    "binary_only_consolidation_"
    "normal_definition_v2"
)

R1_REQUIRED_SOURCE_MARKERS = [
    "NORMAL requires three independent properties",
    "FROZEN NORMAL LOCAL-TIMING REFERENCE",
    "FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE",
    "Coarse semantic information:",
    "Focused semantic information:",
]

R1_FORBIDDEN_SOURCE_MARKERS = [
    "FROZEN NON-NORMAL LOCAL-TIMING REFERENCE PROFILES",
    "Frozen LAG_2 local reference pattern",
    "Frozen LAG_3 local reference pattern",
    "TEMPORAL PROFILE COMPARISON",
    "ORDERED AND INDEPENDENT EVIDENCE ASSESSMENT",
]


# ============================================================
# A1 OUTPUT SCHEMA
# ============================================================

NO_PARTICIPATION_REASONING_OUTPUT_BLOCK = (
    STRUCTURED_REASONING_OUTPUT_BLOCK
    .replace(
        """- participation_assessment:
  - VALID
  - INVALID

""",
        "",
    )
    .replace(
        """  - PARTICIPATION
""",
        "",
    )
    .replace(
        """  "participation_assessment": "VALID or INVALID",
""",
        "",
    )
    .replace(
        (
            '"decisive_dimension": '
            '"PARTICIPATION or TEMPORAL or SEMANTIC or NONE"'
        ),
        (
            '"decisive_dimension": '
            '"TEMPORAL or SEMANTIC or NONE"'
        ),
    )
)

assert (
    "participation_assessment"
    not in NO_PARTICIPATION_REASONING_OUTPUT_BLOCK
)

assert (
    "\n  - PARTICIPATION\n"
    not in NO_PARTICIPATION_REASONING_OUTPUT_BLOCK
)

assert (
    "PARTICIPATION or TEMPORAL"
    not in NO_PARTICIPATION_REASONING_OUTPUT_BLOCK
)


FULL_R1_SCHEMA_KEYS = list(
    REASONING_SCHEMA_KEYS
)

FULL_R1_ALLOWED_VALUES = copy.deepcopy(
    REASONING_ALLOWED_VALUES
)

NO_PARTICIPATION_SCHEMA_KEYS = [
    key
    for key in REASONING_SCHEMA_KEYS
    if key != "participation_assessment"
]

NO_PARTICIPATION_ALLOWED_VALUES = copy.deepcopy(
    REASONING_ALLOWED_VALUES
)

NO_PARTICIPATION_ALLOWED_VALUES.pop(
    "participation_assessment"
)

NO_PARTICIPATION_ALLOWED_VALUES[
    "decisive_dimension"
] = {
    "TEMPORAL",
    "SEMANTIC",
    "NONE",
}


# ============================================================
# EXACT R1 PROMPT LOADING
# ============================================================

def load_exact_full_r1_prompt_templates():
    source_experiment_dir = (
        OUT_DIR
        / R1_SOURCE_EXPERIMENT_NAME
    )

    source_prompt_path = (
        source_experiment_dir
        / "prompt_template.txt"
    )

    assert source_prompt_path.exists(), (
        "Exact R1 source prompt not found:\n"
        f"{source_prompt_path}"
    )

    source_prompt_template = (
        source_prompt_path.read_text(
            encoding="utf-8"
        )
    )

    for marker in R1_REQUIRED_SOURCE_MARKERS:
        assert marker in source_prompt_template, (
            "Required exact-R1 marker missing: "
            f"{marker}"
        )

    for marker in R1_FORBIDDEN_SOURCE_MARKERS:
        assert marker not in source_prompt_template, (
            "Forbidden marker found in exact R1 source: "
            f"{marker}"
        )

    full_r1_reasoning_prompt_template = (
        build_structured_reasoning_prompt_template(
            source_prompt_template
        )
    )

    return {
        "source_prompt_path": source_prompt_path,
        "source_prompt_template": source_prompt_template,
        "full_r1_reasoning_prompt_template": (
            full_r1_reasoning_prompt_template
        ),
    }


# ============================================================
# REMOVE EXACT PROMPT ENTRIES BY PLACEHOLDER
#
# Each requested evidence item is removed at the template level,
# before formatting. Therefore the value is never inserted into
# the model-facing prompt.
# ============================================================

def remove_placeholder_entry(
    prompt_template,
    placeholder_name,
):
    token = (
        "{"
        + placeholder_name
        + "}"
    )

    lines = prompt_template.splitlines()

    hit_indices = [
        index
        for index, line in enumerate(lines)
        if token in line
    ]

    assert len(hit_indices) == 1, (
        f"Expected exactly one {token} placeholder; "
        f"found {len(hit_indices)}"
    )

    hit_index = hit_indices[0]
    start_index = hit_index
    end_index = hit_index

    stripped_hit = (
        lines[hit_index]
        .strip()
        .rstrip(",")
    )

    # If the placeholder is on its own line, also remove the
    # immediately preceding non-empty label line. This avoids
    # leaving a dangling label such as "Filtered turns:".
    if stripped_hit == token:
        previous_index = hit_index - 1

        while (
            previous_index >= 0
            and not lines[previous_index].strip()
        ):
            previous_index -= 1

        if previous_index >= 0:
            previous_line = (
                lines[previous_index]
                .strip()
            )

            is_separator = (
                previous_line
                and set(previous_line)
                <= {
                    "=",
                    "-",
                    "_",
                    "#",
                }
            )

            field_keywords = (
                ["speak"]
                if placeholder_name.endswith(
                    "_speaks"
                )
                else [
                    "turn",
                    "vad",
                ]
            )

            looks_like_target_label = (
                len(previous_line) <= 180
                and "{"
                not in previous_line
                and "}" not in previous_line
                and not is_separator
                and any(
                    keyword
                    in previous_line.lower()
                    for keyword
                    in field_keywords
                )
            )

            if looks_like_target_label:
                start_index = previous_index

    removed_lines = lines[
        start_index:
        end_index + 1
    ]

    kept_lines = (
        lines[:start_index]
        + lines[end_index + 1:]
    )

    # Collapse accidental runs of three or more blank lines.
    cleaned_lines = []

    for line in kept_lines:
        if (
            not line.strip()
            and len(cleaned_lines) >= 2
            and not cleaned_lines[-1].strip()
            and not cleaned_lines[-2].strip()
        ):
            continue

        cleaned_lines.append(
            line
        )

    transformed = "\n".join(
        cleaned_lines
    )

    assert token not in transformed

    return (
        transformed,
        removed_lines,
    )


# ============================================================
# PROMPT-LEVEL ABLATION HELPERS
#
# These functions preserve the exact Structured R1 prompt and
# remove or minimally rewrite only statements that refer to
# evidence unavailable in the current ablation.
# ============================================================

PROMPT_SECTION_RULE = (
    "=" * 60
)


def prompt_section_marker(
    title,
):
    return (
        f"{PROMPT_SECTION_RULE}\n"
        f"{title}\n"
        f"{PROMPT_SECTION_RULE}"
    )


def normalize_prompt_blank_lines(
    text,
):
    while "\n\n\n" in text:
        text = text.replace(
            "\n\n\n",
            "\n\n",
        )

    return text


def replace_exact_prompt_text(
    text,
    old,
    new,
    *,
    description,
):
    count = text.count(
        old
    )

    assert count == 1, (
        f"Expected exactly one prompt fragment for "
        f"{description}; found {count}."
    )

    return text.replace(
        old,
        new,
        1,
    )


def remove_exact_prompt_text(
    text,
    old,
    *,
    description,
):
    return replace_exact_prompt_text(
        text,
        old,
        "",
        description=description,
    )


def locate_prompt_section(
    prompt_template,
    title,
):
    marker = prompt_section_marker(
        title
    )

    start_index = prompt_template.find(
        marker
    )

    assert start_index >= 0, (
        f"Prompt section not found: {title}"
    )

    body_start = (
        start_index
        + len(
            marker
        )
    )

    next_section_index = prompt_template.find(
        PROMPT_SECTION_RULE,
        body_start,
    )

    assert next_section_index >= 0, (
        f"No following section found after: {title}"
    )

    return {
        "marker": marker,
        "start": start_index,
        "body_start": body_start,
        "next_start": next_section_index,
        "body": prompt_template[
            body_start:
            next_section_index
        ].strip(),
    }


def replace_prompt_section_body(
    prompt_template,
    title,
    new_body,
):
    section = locate_prompt_section(
        prompt_template,
        title,
    )

    replacement = (
        section[
            "marker"
        ]
        + "\n\n"
        + new_body.strip()
        + "\n\n"
    )

    transformed = (
        prompt_template[
            :section[
                "start"
            ]
        ]
        + replacement
        + prompt_template[
            section[
                "next_start"
            ]:
        ]
    )

    return normalize_prompt_blank_lines(
        transformed
    )


def remove_prompt_section(
    prompt_template,
    title,
):
    section = locate_prompt_section(
        prompt_template,
        title,
    )

    transformed = (
        prompt_template[
            :section[
                "start"
            ]
        ]
        + prompt_template[
            section[
                "next_start"
            ]:
        ]
    )

    return normalize_prompt_blank_lines(
        transformed
    )


def build_available_evidence_section_body(
    *,
    include_speaks,
    include_filtered_turns,
):
    evidence_items = []

    if include_speaks:
        evidence_items.append(
            "Whether each participant speaks during the "
            "120-second interval."
        )

    if include_filtered_turns:
        evidence_items.append(
            "Independently backchannel-filtered speech turns "
            "for each participant."
        )

    evidence_items.extend(
        [
            "Local turn-handoff and overlap features.",
            "Global temporal alignment-shift features.",
            (
                "Coarse semantic summaries for two "
                "synchronized 60-second segments."
            ),
            (
                "Focused semantic summaries for the same "
                "two segments."
            ),
            (
                "Frozen temporal reference statistics calculated only from\n"
                "   separate NORMAL dyadic conversations."
            ),
        ]
    )

    numbered_items = "\n".join(
        f"{index}. {item}"
        for index, item in enumerate(
            evidence_items,
            start=1,
        )
    )

    return (
        "You receive:\n\n"
        + numbered_items
        + "\n\n"
        + "Use only the supplied evidence.\n\n"
        + (
            "Do not infer anything from identifiers, filenames, paths, "
            "dataset order,\n"
            "sample position, participant identity, or hidden labels.\n\n"
            "None of those fields are provided."
        )
    )


def adapt_operational_definition_without_participation(
    prompt_template,
):
    title = (
        "OPERATIONAL DEFINITION OF NORMAL"
    )

    section = locate_prompt_section(
        prompt_template,
        title,
    )

    body = section[
        "body"
    ]

    body = replace_exact_prompt_text(
        body,
        "NORMAL requires three independent properties:",
        "NORMAL requires two independent properties:",
        description=(
            "three-to-two property count"
        ),
    )

    body = replace_exact_prompt_text(
        body,
        (
            "1. Participation validity\n"
            "2. Semantic conversational compatibility\n"
            "3. Temporal coordination"
        ),
        (
            "1. Semantic conversational compatibility\n"
            "2. Temporal coordination"
        ),
        description=(
            "operational property list"
        ),
    )

    body = replace_exact_prompt_text(
        body,
        (
            "Evaluate these three properties separately before making "
            "the final\n"
            "binary decision."
        ),
        (
            "Evaluate these two properties separately before making "
            "the final\n"
            "binary decision."
        ),
        description=(
            "operational evaluation count"
        ),
    )

    body = replace_exact_prompt_text(
        body,
        (
            "A case should be classified as NORMAL only when all three "
            "properties\n"
            "are sufficiently supported by the available evidence."
        ),
        (
            "A case should be classified as NORMAL only when both "
            "properties\n"
            "are sufficiently supported by the available evidence."
        ),
        description=(
            "operational all-to-both rule"
        ),
    )

    body = replace_exact_prompt_text(
        body,
        (
            "A strong and reliable failure of any one property means "
            "that the\n"
            "complete interaction does not satisfy the operational "
            "definition of\n"
            "NORMAL."
        ),
        (
            "A strong and reliable failure of either property means "
            "that the\n"
            "complete interaction does not satisfy the operational "
            "definition of\n"
            "NORMAL."
        ),
        description=(
            "operational failure rule"
        ),
    )

    body = remove_exact_prompt_text(
        body,
        (
            "Do not use a simple majority vote between the three "
            "properties.\n\n"
        ),
        description=(
            "three-property majority sentence"
        ),
    )

    body = replace_exact_prompt_text(
        body,
        (
            "Evidence that two properties appear normal must not "
            "override a strong\n"
            "and reliable failure of the third property."
        ),
        (
            "Evidence that one property appears normal must not "
            "override a strong\n"
            "and reliable failure of the other property."
        ),
        description=(
            "two-property non-cancellation rule"
        ),
    )

    body = remove_exact_prompt_text(
        body,
        (
            "- Both participants speaking does not by itself prove "
            "NORMAL.\n"
        ),
        description=(
            "participation bullet in operational definition"
        ),
    )

    return replace_prompt_section_body(
        prompt_template,
        title,
        body,
    )


def adapt_participation_validity_section(
    prompt_template,
    *,
    include_speaks,
    include_filtered_turns,
):
    title = (
        "PARTICIPATION VALIDITY"
    )

    if (
        not include_speaks
        and not include_filtered_turns
    ):
        return remove_prompt_section(
            prompt_template,
            title,
        )

    section = locate_prompt_section(
        prompt_template,
        title,
    )

    body = section[
        "body"
    ]

    speaks_definition = (
        "The field \"speaks\" is derived directly from the final "
        "filtered VAD turns:\n\n"
        "- speaks = true means that the participant has at least one "
        "retained turn.\n"
        "- speaks = false means that the participant has no retained "
        "turns during\n"
        "  the entire 120-second interval.\n\n"
    )

    joint_instruction = (
        "Use the speaks fields together with the actual filtered turn "
        "lists."
    )

    if (
        not include_speaks
        and include_filtered_turns
    ):
        body = remove_exact_prompt_text(
            body,
            speaks_definition,
            description=(
                "speaks definition"
            ),
        )

        body = replace_exact_prompt_text(
            body,
            joint_instruction,
            "Use the actual filtered turn lists.",
            description=(
                "turn-only participation instruction"
            ),
        )

        body = replace_exact_prompt_text(
            body,
            (
                "However, the complete absence of retained speech from "
                "one participant\n"
                "is not compatible with a normal two-person spoken "
                "interaction."
            ),
            (
                "However, an empty filtered turn list for one participant "
                "is not\n"
                "compatible with a normal two-person spoken interaction."
            ),
            description=(
                "turn-only absence rule"
            ),
        )

    elif (
        include_speaks
        and not include_filtered_turns
    ):
        body = replace_exact_prompt_text(
            body,
            joint_instruction,
            "Use the speaks fields.",
            description=(
                "speaks-only participation instruction"
            ),
        )

    return replace_prompt_section_body(
        prompt_template,
        title,
        body,
    )


def remove_turn_specific_prompt_instructions(
    prompt_template,
):
    prompt_template = remove_prompt_section(
        prompt_template,
        "TURN FORMAT AND BACKCHANNEL FILTERING",
    )

    prompt_template = remove_exact_prompt_text(
        prompt_template,
        (
            "- repeated delayed handoffs visible in the raw turn "
            "structure.\n"
        ),
        description=(
            "raw-turn local-reference bullet"
        ),
    )

    prompt_template = remove_exact_prompt_text(
        prompt_template,
        "- the raw turn structure,\n",
        description=(
            "raw-turn overlap bullet"
        ),
    )

    prompt_template = replace_exact_prompt_text(
        prompt_template,
        "- and the global alignment evidence.",
        "- the global alignment evidence.",
        description=(
            "reduced overlap-list grammar"
        ),
    )

    prompt_template = remove_exact_prompt_text(
        prompt_template,
        (
            "The observed Participant B turns supplied for classification "
            "are not\n"
            "changed.\n\n"
        ),
        description=(
            "observed-turn global-alignment statement"
        ),
    )

    return normalize_prompt_blank_lines(
        prompt_template
    )


def adapt_joint_temporal_section_without_participation(
    prompt_template,
):
    title = (
        "JOINT TEMPORAL COORDINATION DECISION"
    )

    section = locate_prompt_section(
        prompt_template,
        title,
    )

    body = remove_exact_prompt_text(
        section[
            "body"
        ],
        "- both participants speak,\n",
        description=(
            "participation bullet in temporal decision"
        ),
    )

    return replace_prompt_section_body(
        prompt_template,
        title,
        body,
    )


def adapt_final_decision_without_participation(
    prompt_template,
):
    title = (
        "FINAL COMBINED DECISION"
    )

    section = locate_prompt_section(
        prompt_template,
        title,
    )

    body = section[
        "body"
    ]

    body = replace_exact_prompt_text(
        body,
        (
            "Internally evaluate:\n\n"
            "1. Participation validity\n"
            "2. Semantic conversational compatibility\n"
            "3. Temporal coordination"
        ),
        (
            "Internally evaluate:\n\n"
            "1. Semantic conversational compatibility\n"
            "2. Temporal coordination"
        ),
        description=(
            "final decision property list"
        ),
    )

    body = replace_exact_prompt_text(
        body,
        (
            "Classify as NORMAL only when the complete evidence is "
            "sufficiently\n"
            "compatible with all three required properties of a normal "
            "dyadic\n"
            "interaction."
        ),
        (
            "Classify as NORMAL only when the complete evidence is "
            "sufficiently\n"
            "compatible with both required properties of a normal "
            "dyadic\n"
            "interaction."
        ),
        description=(
            "final all-to-both rule"
        ),
    )

    body = remove_exact_prompt_text(
        body,
        (
            "- Speech from both participants must not cancel semantic "
            "or temporal failure.\n"
        ),
        description=(
            "participation bullet in final decision"
        ),
    )

    return replace_prompt_section_body(
        prompt_template,
        title,
        body,
    )


def remove_empty_current_case_participant_blocks(
    prompt_template,
):
    current_case_marker = prompt_section_marker(
        "CURRENT CASE"
    )

    current_case_start = prompt_template.find(
        current_case_marker
    )

    assert current_case_start >= 0

    participant_start = prompt_template.find(
        "PARTICIPANT A",
        current_case_start,
    )

    local_features_start = prompt_template.find(
        "LOCAL TEMPORAL FEATURES",
        participant_start,
    )

    assert participant_start >= 0
    assert local_features_start >= 0

    transformed = (
        prompt_template[
            :participant_start
        ]
        + prompt_template[
            local_features_start:
        ]
    )

    return normalize_prompt_blank_lines(
        transformed
    )


def audit_participation_ablation_prompt_template(
    prompt_template,
    *,
    include_speaks,
    include_filtered_turns,
    include_participation_assessment,
):
    if include_speaks:
        assert "{participant_A_speaks}" in prompt_template
        assert "{participant_B_speaks}" in prompt_template
        assert (
            "Whether each participant speaks during the "
            "120-second interval."
            in prompt_template
        )
    else:
        assert "{participant_A_speaks}" not in prompt_template
        assert "{participant_B_speaks}" not in prompt_template
        assert (
            "Whether each participant speaks during the "
            "120-second interval."
            not in prompt_template
        )
        assert "speaks = true" not in prompt_template
        assert "speaks = false" not in prompt_template

    if include_filtered_turns:
        assert "{participant_A_turns}" in prompt_template
        assert "{participant_B_turns}" in prompt_template
        assert (
            "Independently backchannel-filtered speech turns "
            "for each participant."
            in prompt_template
        )
        assert (
            prompt_section_marker(
                "TURN FORMAT AND BACKCHANNEL FILTERING"
            )
            in prompt_template
        )
    else:
        assert "{participant_A_turns}" not in prompt_template
        assert "{participant_B_turns}" not in prompt_template
        assert (
            "Independently backchannel-filtered speech turns "
            "for each participant."
            not in prompt_template
        )
        assert (
            prompt_section_marker(
                "TURN FORMAT AND BACKCHANNEL FILTERING"
            )
            not in prompt_template
        )
        assert "raw turn structure" not in prompt_template
        assert (
            "Participant B turns supplied for classification"
            not in prompt_template
        )

    if include_participation_assessment:
        assert (
            prompt_section_marker(
                "PARTICIPATION VALIDITY"
            )
            in prompt_template
        )
        assert (
            "participation_assessment"
            in prompt_template
        )
    else:
        assert (
            prompt_section_marker(
                "PARTICIPATION VALIDITY"
            )
            not in prompt_template
        )
        assert "Participation validity" not in prompt_template
        assert "participation_assessment" not in prompt_template
        assert "PARTICIPATION or TEMPORAL" not in prompt_template
        assert "NORMAL requires two independent properties" in prompt_template
        assert "NORMAL requires three independent properties" not in prompt_template
        assert "both required properties" in prompt_template
        assert "all three required properties" not in prompt_template
        assert "both participants speak" not in prompt_template
        assert "Both participants speaking" not in prompt_template
        assert "Speech from both participants" not in prompt_template

    # Core R1 branches that must remain in every participation ablation.
    for required_marker in [
        "LOCAL TEMPORAL FEATURES",
        "FROZEN NORMAL LOCAL-TIMING REFERENCE",
        "GLOBAL ALIGNMENT-SHIFT FEATURES",
        "FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE",
        "JOINT TEMPORAL COORDINATION DECISION",
        "SEMANTIC EVIDENCE",
        "FINAL COMBINED DECISION",
        "STRUCTURED REASONING OUTPUT",
    ]:
        assert required_marker in prompt_template


def build_participation_ablation_prompt_template(
    full_r1_prompt_template,
    *,
    include_speaks,
    include_filtered_turns,
    include_participation_assessment,
):
    prompt_template = (
        full_r1_prompt_template
    )

    removed_template_entries = {}

    # --------------------------------------------------------
    # 1. Remove unavailable CURRENT CASE values/placeholders.
    # --------------------------------------------------------
    if not include_speaks:
        for role in [
            "participant_A",
            "participant_B",
        ]:
            placeholder_name = (
                f"{role}_speaks"
            )

            (
                prompt_template,
                removed_lines,
            ) = remove_placeholder_entry(
                prompt_template,
                placeholder_name,
            )

            removed_template_entries[
                placeholder_name
            ] = removed_lines

    if not include_filtered_turns:
        for role in [
            "participant_A",
            "participant_B",
        ]:
            placeholder_name = (
                f"{role}_turns"
            )

            (
                prompt_template,
                removed_lines,
            ) = remove_placeholder_entry(
                prompt_template,
                placeholder_name,
            )

            removed_template_entries[
                placeholder_name
            ] = removed_lines

    # --------------------------------------------------------
    # 2. Rebuild AVAILABLE EVIDENCE using only supplied inputs.
    # --------------------------------------------------------
    prompt_template = replace_prompt_section_body(
        prompt_template,
        "AVAILABLE EVIDENCE",
        build_available_evidence_section_body(
            include_speaks=include_speaks,
            include_filtered_turns=(
                include_filtered_turns
            ),
        ),
    )

    removed_template_entries[
        "available_evidence_adaptation"
    ] = {
        "include_speaks": bool(
            include_speaks
        ),
        "include_filtered_turns": bool(
            include_filtered_turns
        ),
    }

    # --------------------------------------------------------
    # 3. Adapt or remove the participation reasoning branch.
    # --------------------------------------------------------
    if not include_participation_assessment:
        prompt_template = (
            adapt_operational_definition_without_participation(
                prompt_template
            )
        )

    prompt_template = adapt_participation_validity_section(
        prompt_template,
        include_speaks=include_speaks,
        include_filtered_turns=(
            include_filtered_turns
        ),
    )

    if not include_participation_assessment:
        prompt_template = (
            adapt_joint_temporal_section_without_participation(
                prompt_template
            )
        )

        prompt_template = (
            adapt_final_decision_without_participation(
                prompt_template
            )
        )

        removed_template_entries[
            "participation_branch"
        ] = [
            "PARTICIPATION VALIDITY section",
            "participation property in operational definition",
            "participation property in final decision",
            "participation-dependent cross-branch bullets",
            "participation_assessment output",
            "PARTICIPATION decisive_dimension option",
        ]

    elif (
        not include_speaks
        and include_filtered_turns
    ):
        removed_template_entries[
            "participation_instruction_adaptation"
        ] = [
            "Removed speaks-field definition",
            "Participation assessment now uses filtered turns only",
        ]

    elif (
        include_speaks
        and not include_filtered_turns
    ):
        removed_template_entries[
            "participation_instruction_adaptation"
        ] = [
            "Participation assessment now uses speaks fields only",
        ]

    # --------------------------------------------------------
    # 4. Remove instructions that require raw filtered turns.
    # --------------------------------------------------------
    if not include_filtered_turns:
        prompt_template = (
            remove_turn_specific_prompt_instructions(
                prompt_template
            )
        )

        removed_template_entries[
            "filtered_turn_instruction_removals"
        ] = [
            "TURN FORMAT AND BACKCHANNEL FILTERING section",
            "raw-turn-structure local-reference bullet",
            "raw-turn-structure overlap bullet",
            "observed Participant B turns statement",
        ]

    # If neither participant feature remains, remove the now-empty
    # PARTICIPANT A / PARTICIPANT B blocks from CURRENT CASE.
    if (
        not include_speaks
        and not include_filtered_turns
    ):
        prompt_template = (
            remove_empty_current_case_participant_blocks(
                prompt_template
            )
        )

        removed_template_entries[
            "current_case_participant_blocks"
        ] = [
            "PARTICIPANT A empty block",
            "PARTICIPANT B empty block",
        ]

    # --------------------------------------------------------
    # 5. Adapt the structured output only for full A1.
    # --------------------------------------------------------
    if include_participation_assessment:
        assert prompt_template.endswith(
            STRUCTURED_REASONING_OUTPUT_BLOCK
        )
    else:
        assert prompt_template.endswith(
            STRUCTURED_REASONING_OUTPUT_BLOCK
        )

        prompt_template = (
            prompt_template[
                :-len(
                    STRUCTURED_REASONING_OUTPUT_BLOCK
                )
            ]
            + NO_PARTICIPATION_REASONING_OUTPUT_BLOCK
        )

    prompt_template = normalize_prompt_blank_lines(
        prompt_template
    )

    audit_participation_ablation_prompt_template(
        prompt_template,
        include_speaks=include_speaks,
        include_filtered_turns=(
            include_filtered_turns
        ),
        include_participation_assessment=(
            include_participation_assessment
        ),
    )

    return (
        prompt_template,
        removed_template_entries,
    )


# ============================================================
# ABLATED INPUT PROJECTION
# ============================================================

def build_participation_ablation_model_input(
    case,
    *,
    include_speaks,
    include_filtered_turns,
):
    full_payload = (
        build_binary_model_input(
            case
        )
    )

    ablated_payload = copy.deepcopy(
        full_payload
    )

    for role in [
        "participant_A",
        "participant_B",
    ]:
        if not include_speaks:
            ablated_payload[
                role
            ].pop(
                "speaks"
            )

        if not include_filtered_turns:
            ablated_payload[
                role
            ].pop(
                "filtered_turns"
            )

    return (
        ablated_payload,
        full_payload,
    )


def build_participation_ablation_prompt(
    case,
    config,
):
    (
        ablated_payload,
        full_payload,
    ) = build_participation_ablation_model_input(
        case,
        include_speaks=(
            config[
                "include_speaks"
            ]
        ),
        include_filtered_turns=(
            config[
                "include_filtered_turns"
            ]
        ),
    )

    prompt = (
        config[
            "reasoning_prompt_template"
        ]
        .format(
            normal_base_reference_text=(
                normal_base_reference_text
            ),

            normal_global_reference_text=(
                normal_global_reference_text
            ),

            duration_seconds=(
                full_payload[
                    "analysis_duration_seconds"
                ]
            ),

            participant_A_speaks=(
                canonical_json(
                    full_payload[
                        "participant_A"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_A_turns=(
                canonical_json(
                    full_payload[
                        "participant_A"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            participant_B_speaks=(
                canonical_json(
                    full_payload[
                        "participant_B"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_B_turns=(
                canonical_json(
                    full_payload[
                        "participant_B"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            local_temporal_features=(
                json.dumps(
                    full_payload[
                        "local_temporal_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            global_shift_features=(
                json.dumps(
                    full_payload[
                        "global_shift_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            semantic_summaries=(
                json.dumps(
                    full_payload[
                        "semantic_summaries"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),
        )
    )

    prompt_lower = prompt.lower()

    for forbidden_key in (
        FORBIDDEN_PROMPT_KEYS
    ):
        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )

    return (
        prompt,
        ablated_payload,
    )


# ============================================================
# CONFIGURATION + ARTIFACT AUDIT
# ============================================================

def prepare_participation_ablation_experiment(
    *,
    experiment_version,
    experiment_title,
    ablation_id,
    include_speaks,
    include_filtered_turns,
    include_participation_assessment,
):
    exact_r1 = (
        load_exact_full_r1_prompt_templates()
    )

    source_prompt_template = (
        exact_r1[
            "source_prompt_template"
        ]
    )

    full_r1_prompt_template = (
        exact_r1[
            "full_r1_reasoning_prompt_template"
        ]
    )

    (
        ablation_prompt_template,
        removed_template_entries,
    ) = build_participation_ablation_prompt_template(
        full_r1_prompt_template,
        include_speaks=include_speaks,
        include_filtered_turns=(
            include_filtered_turns
        ),
        include_participation_assessment=(
            include_participation_assessment
        ),
    )

    if include_participation_assessment:
        schema_keys = list(
            FULL_R1_SCHEMA_KEYS
        )

        allowed_values = copy.deepcopy(
            FULL_R1_ALLOWED_VALUES
        )

        output_block = (
            STRUCTURED_REASONING_OUTPUT_BLOCK
        )

    else:
        schema_keys = list(
            NO_PARTICIPATION_SCHEMA_KEYS
        )

        allowed_values = copy.deepcopy(
            NO_PARTICIPATION_ALLOWED_VALUES
        )

        output_block = (
            NO_PARTICIPATION_REASONING_OUTPUT_BLOCK
        )

    experiment_dir = (
        OUT_DIR
        / experiment_version
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),

        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),

        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),

        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),

        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),

        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),

        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),

        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),

        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),

        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),

        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),

        "source_prompt_copy": (
            experiment_dir
            / "source_binary_prompt_template.txt"
        ),

        "full_r1_prompt_copy": (
            experiment_dir
            / "full_structured_r1_prompt_template.txt"
        ),

        "prompt_diff": (
            experiment_dir
            / "prompt_diff_vs_full_structured_r1.txt"
        ),

        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }

    paths[
        "prompt_template"
    ].write_text(
        ablation_prompt_template,
        encoding="utf-8",
    )

    paths[
        "source_prompt_copy"
    ].write_text(
        source_prompt_template,
        encoding="utf-8",
    )

    paths[
        "full_r1_prompt_copy"
    ].write_text(
        full_r1_prompt_template,
        encoding="utf-8",
    )

    prompt_diff_lines = (
        difflib.unified_diff(
            full_r1_prompt_template.splitlines(),
            ablation_prompt_template.splitlines(),
            fromfile=(
                "full_structured_r1_prompt_template.txt"
            ),
            tofile="prompt_template.txt",
            lineterm="",
        )
    )

    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(
            prompt_diff_lines
        ),
        encoding="utf-8",
    )

    source_prompt_sha256 = (
        sha256_text(
            source_prompt_template
        )
    )

    full_r1_prompt_sha256 = (
        sha256_text(
            full_r1_prompt_template
        )
    )

    reasoning_prompt_sha256 = (
        sha256_text(
            ablation_prompt_template
        )
    )

    reasoning_schema_sha256 = (
        sha256_text(
            output_block
        )
    )

    normal_reference_sha256 = (
        sha256_text(
            normal_base_reference_text
            + "\n"
            + normal_global_reference_text
        )
    )

    temporary_config = {
        "include_speaks": (
            include_speaks
        ),

        "include_filtered_turns": (
            include_filtered_turns
        ),

        "reasoning_prompt_template": (
            ablation_prompt_template
        ),
    }

    (
        example_prompt,
        example_payload,
    ) = build_participation_ablation_prompt(
        consolidation_cases[0],
        temporary_config,
    )

    payload_keys = (
        collect_nested_keys_reasoning(
            example_payload
        )
    )

    for required_key in (
        FULL_SEMANTIC_REQUIRED_KEYS
    ):
        assert required_key in payload_keys, (
            "Full semantic evidence changed or missing: "
            f"{required_key}"
        )

    for required_key in [
        "local_temporal_features",
        "global_shift_features",
        "semantic_summaries",
    ]:
        assert required_key in payload_keys

    for role in [
        "participant_A",
        "participant_B",
    ]:
        role_payload = (
            example_payload[
                role
            ]
        )

        assert (
            ("speaks" in role_payload)
            == include_speaks
        )

        assert (
            ("filtered_turns" in role_payload)
            == include_filtered_turns
        )

    if include_participation_assessment:
        assert (
            "participation_assessment"
            in ablation_prompt_template
        )
    else:
        assert (
            "participation_assessment"
            not in ablation_prompt_template
        )

        assert (
            "PARTICIPATION or TEMPORAL"
            not in ablation_prompt_template
        )

    manifest = {
        "experiment_version": (
            experiment_version
        ),

        "experiment_title": (
            experiment_title
        ),

        "ablation_id": (
            ablation_id
        ),

        "source_experiment": (
            R1_SOURCE_EXPERIMENT_NAME
        ),

        "source_prompt_path": str(
            exact_r1[
                "source_prompt_path"
            ]
        ),

        "source_prompt_sha256": (
            source_prompt_sha256
        ),

        "full_r1_prompt_sha256": (
            full_r1_prompt_sha256
        ),

        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),

        "reasoning_schema_sha256": (
            reasoning_schema_sha256
        ),

        "normal_reference_sha256": (
            normal_reference_sha256
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": [
            "NORMAL"
        ],

        "assessment_policy": (
            "Exact Structured R1 policy with prompt-consistent "
            "removal of only the declared unavailable participation "
            "evidence and its dependent instructions."
        ),

        "include_speaks": bool(
            include_speaks
        ),

        "include_filtered_turns": bool(
            include_filtered_turns
        ),

        "include_participation_assessment": bool(
            include_participation_assessment
        ),

        "schema_keys": (
            schema_keys
        ),

        "allowed_values": {
            key: sorted(
                list(
                    values
                )
            )
            for key, values
            in allowed_values.items()
        },

        "removed_template_entries": (
            removed_template_entries
        ),

        "unchanged_evidence": [
            "analysis_duration_seconds",
            "local_temporal_features",
            "global_shift_features",
            "coarse_semantic_summaries",
            "focused_semantic_summaries",
            "frozen_NORMAL_references",
        ],

        "model_id": (
            MODEL_ID
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },
    }

    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    config = {
        **manifest,

        "experiment_dir": (
            experiment_dir
        ),

        "source_prompt_template": (
            source_prompt_template
        ),

        "full_r1_prompt_template": (
            full_r1_prompt_template
        ),

        "reasoning_prompt_template": (
            ablation_prompt_template
        ),

        "schema_keys": (
            schema_keys
        ),

        "allowed_values": (
            allowed_values
        ),

        "paths": (
            paths
        ),
    }

    print("=" * 88)
    print(
        f"{experiment_title} — CONFIGURATION READY"
    )
    print("=" * 88)

    print(
        "Ablation ID:",
        ablation_id,
    )

    print(
        "Include participant speaks:",
        include_speaks,
    )

    print(
        "Include filtered turns:",
        include_filtered_turns,
    )

    print(
        "Include participation_assessment:",
        include_participation_assessment,
    )

    print(
        "Schema keys:",
        schema_keys,
    )

    print(
        "Full R1 prompt SHA256:",
        full_r1_prompt_sha256,
    )

    print(
        "Ablation prompt SHA256:",
        reasoning_prompt_sha256,
    )

    print(
        "Prompt diff:",
        paths[
            "prompt_diff"
        ],
    )

    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )

    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )

    return config


# ============================================================
# PROMPT INSPECTION GATE
#
# The run function refuses to start until the exact rendered
# prompt for that configuration has been printed in this runtime.
# ============================================================

INSPECTED_PARTICIPATION_ABLATIONS = set()


def inspect_participation_ablation_prompt(
    config,
    *,
    case_index=0,
):
    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert (
        0 <= case_index < len(
            ordered_cases
        )
    )

    case = ordered_cases[
        case_index
    ]

    (
        prompt,
        payload,
    ) = build_participation_ablation_prompt(
        case,
        config,
    )

    print("=" * 100)
    print(
        "PROMPT INSPECTION —",
        config[
            "experiment_title"
        ],
    )
    print("=" * 100)

    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )

    print(
        "Gold label is intentionally NOT printed "
        "inside the model prompt."
    )

    print("\nREMOVED / ADAPTED PROMPT COMPONENTS")

    print(
        json.dumps(
            config[
                "removed_template_entries"
            ],
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\nEXACT ABLATED MODEL INPUT PAYLOAD")

    print(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\n" + "=" * 100)
    print("EXACT RENDERED MODEL PROMPT")
    print("=" * 100)

    print(
        prompt
    )

    print("\n" + "=" * 100)
    print("PROMPT INSPECTION HASHES")
    print("=" * 100)

    print(
        "Rendered prompt SHA256:",
        sha256_text(
            prompt
        ),
    )

    print(
        "Ablated payload SHA256:",
        sha256_text(
            canonical_json(
                payload
            )
        ),
    )

    print(
        "Template diff file:",
        config[
            "paths"
        ][
            "prompt_diff"
        ],
    )

    INSPECTED_PARTICIPATION_ABLATIONS.add(
        config[
            "experiment_version"
        ]
    )

    return {
        "case_id": str(
            case[
                "case_id"
            ]
        ),
        "prompt": prompt,
        "payload": payload,
    }


# ============================================================
# CONFIG-AWARE STRUCTURED PARSER
# ============================================================

STANDARD_REASONING_FIELDS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


def parse_participation_ablation_prediction(
    raw_output,
    config,
):
    parsed = (
        extract_first_json_object(
            raw_output
        )
    )

    schema_keys = (
        config[
            "schema_keys"
        ]
    )

    allowed_values = (
        config[
            "allowed_values"
        ]
    )

    normalized = {
        key: None
        for key in schema_keys
    }

    schema_errors = []

    if isinstance(
        parsed,
        dict,
    ):
        for key in schema_keys:
            if key in parsed:
                normalized[
                    key
                ] = str(
                    parsed[
                        key
                    ]
                ).strip().upper()

        for key in schema_keys:
            if normalized[
                key
            ] not in allowed_values[
                key
            ]:
                schema_errors.append(
                    f"{key}: {normalized[key]}"
                )

        exact_keys = (
            set(
                parsed.keys()
            )
            == set(
                schema_keys
            )
        )

        schema_exact = (
            exact_keys
            and not schema_errors
        )

        label = normalized[
            "label"
        ]

        if label in LABELS:
            result = {
                "prediction": label,
                "parse_mode": (
                    "structured_json"
                ),
                "schema_exact": bool(
                    schema_exact
                ),
                "parsed_output": parsed,
                "schema_errors": (
                    schema_errors
                ),
            }

            for field in (
                STANDARD_REASONING_FIELDS
            ):
                result[
                    field
                ] = normalized.get(
                    field
                )

            return result

    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )

    if plain_output in LABELS:
        return {
            "prediction": plain_output,
            "parse_mode": (
                "exact_plaintext_fallback"
            ),
            "schema_exact": False,
            "parsed_output": None,
            "schema_errors": [
                "Structured reasoning fields missing."
            ],
            **{
                field: None
                for field in (
                    STANDARD_REASONING_FIELDS
                )
            },
        }

    return {
        "prediction": None,
        "parse_mode": "invalid",
        "schema_exact": False,
        "parsed_output": parsed,
        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid structured JSON prediction."
            ]
        ),
        **{
            field: normalized.get(
                field
            )
            for field in (
                STANDARD_REASONING_FIELDS
            )
        },
    }


# ============================================================
# CHECKPOINT CACHE
# ============================================================

def create_participation_ablation_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),

        "ablation_id": (
            config[
                "ablation_id"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "model_id": (
            MODEL_ID
        ),

        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),

        "full_r1_prompt_sha256": (
            config[
                "full_r1_prompt_sha256"
            ]
        ),

        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),

        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),

        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": [
            "NORMAL"
        ],

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "include_speaks": (
            config[
                "include_speaks"
            ]
        ),

        "include_filtered_turns": (
            config[
                "include_filtered_turns"
            ]
        ),

        "include_participation_assessment": (
            config[
                "include_participation_assessment"
            ]
        ),

        "schema_keys": (
            config[
                "schema_keys"
            ]
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "created_at_utc": (
            reasoning_utc_now()
        ),

        "updated_at_utc": (
            reasoning_utc_now()
        ),

        "records": {},
    }


def run_participation_ablation_experiment(
    config,
    *,
    print_each_case_prompt=False,
):
    assert (
        config[
            "experiment_version"
        ]
        in INSPECTED_PARTICIPATION_ABLATIONS
    ), (
        "Prompt inspection has not been completed in this "
        "runtime. Run the inspection cell immediately above "
        "before starting inference."
    )

    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )

    expected_cache_header = (
        create_participation_ablation_cache(
            config
        )
    )

    if prediction_cache_path.exists():
        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )

        for key in [
            "experiment_version",
            "ablation_id",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "full_r1_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "temporal_profiles_used",
            "assessment_policy",
            "include_speaks",
            "include_filtered_turns",
            "include_participation_assessment",
            "schema_keys",
            "max_new_tokens",
        ]:
            assert (
                prediction_cache[
                    key
                ]
                == expected_cache_header[
                    key
                ]
            ), (
                f"Cache mismatch for {key}"
            )

        print(
            "Resuming cache:",
            prediction_cache_path,
        )

        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )

    else:
        prediction_cache = (
            expected_cache_header
        )

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

        print(
            "Created cache:",
            prediction_cache_path,
        )

    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert len(
        ordered_cases
    ) == 400

    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):
        case_id = str(
            case[
                "case_id"
            ]
        )

        (
            prompt,
            model_input_payload,
        ) = build_participation_ablation_prompt(
            case,
            config,
        )

        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )

        for required_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):
            assert required_key in (
                current_payload_keys
            ), (
                f"Missing full-semantic field "
                f"for case {case_id}: "
                f"{required_key}"
            )

        for role in [
            "participant_A",
            "participant_B",
        ]:
            role_payload = (
                model_input_payload[
                    role
                ]
            )

            assert (
                ("speaks" in role_payload)
                == config[
                    "include_speaks"
                ]
            )

            assert (
                ("filtered_turns" in role_payload)
                == config[
                    "include_filtered_turns"
                ]
            )

        prompt_sha256 = (
            sha256_text(
                prompt
            )
        )

        input_payload_sha256 = (
            sha256_text(
                canonical_json(
                    model_input_payload
                )
            )
        )

        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )

        if (
            existing_record is not None
            and existing_record.get(
                "prediction"
            ) in LABELS
        ):
            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )

            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )

            continue

        if print_each_case_prompt:
            print("\n" + "=" * 100)
            print(
                "CASE:",
                case_id,
            )
            print("=" * 100)
            print(
                prompt
            )

        started = (
            time.perf_counter()
        )

        try:
            (
                raw_output,
                input_token_count,
            ) = qwen_text_only_binary(
                prompt,
                max_new_tokens=(
                    MAX_NEW_TOKENS_REASONING
                ),
            )

            parsed_result = (
                parse_participation_ablation_prediction(
                    raw_output,
                    config,
                )
            )

            generation_error = None

        except Exception as exc:
            raw_output = ""
            input_token_count = None

            parsed_result = {
                "prediction": None,
                "parse_mode": (
                    "generation_error"
                ),
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                **{
                    field: None
                    for field in (
                        STANDARD_REASONING_FIELDS
                    )
                },
            }

            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        elapsed_seconds = (
            time.perf_counter()
            - started
        )

        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "ablation_id": (
                config[
                    "ablation_id"
                ]
            ),

            "include_speaks": (
                config[
                    "include_speaks"
                ]
            ),

            "include_filtered_turns": (
                config[
                    "include_filtered_turns"
                ]
            ),

            "include_participation_assessment": (
                config[
                    "include_participation_assessment"
                ]
            ),

            "prompt_sha256": (
                prompt_sha256
            ),

            "input_payload_sha256": (
                input_payload_sha256
            ),

            "semantic_input": (
                "coarse_and_focused"
            ),

            "focused_summaries_used": True,

            "input_token_count": (
                input_token_count
            ),

            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),

            "raw_output": (
                raw_output
            ),

            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),

            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),

            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),

            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),

            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),

            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),

            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),

            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),

            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),

            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),

            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),

            "generation_error": (
                generation_error
            ),

            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),

            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }

        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)

    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )

    print(
        "Prediction cache:",
        prediction_cache_path,
    )

    return prediction_cache


## Configure all three participation ablations

This cell creates three separate output directories and caches. It does not run inference.

In [ ]:

# ============================================================
# PHASE A — PARTICIPATION BRANCH ABLATIONS
#
# A1   : remove speaks + filtered turns + participation output
# A1-S : remove speaks only; keep filtered turns and full R1 output
# A1-T : remove filtered turns only; keep speaks and full R1 output
# ============================================================

A1_NO_PARTICIPATION_CONFIG = (
    prepare_participation_ablation_experiment(
        experiment_version=(
            "ablation_a1_no_participation_branch"
        ),

        experiment_title=(
            "A1 — No Participation Branch"
        ),

        ablation_id=(
            "A1_NO_PARTICIPATION_BRANCH"
        ),

        include_speaks=False,

        include_filtered_turns=False,

        include_participation_assessment=False,
    )
)


A1_NO_SPEAKS_CONFIG = (
    prepare_participation_ablation_experiment(
        experiment_version=(
            "ablation_a1_no_speaks_keep_turns"
        ),

        experiment_title=(
            "A1-S — No Speaks, Keep Filtered Turns"
        ),

        ablation_id=(
            "A1_NO_SPEAKS_KEEP_TURNS"
        ),

        include_speaks=False,

        include_filtered_turns=True,

        include_participation_assessment=True,
    )
)


A1_NO_TURNS_CONFIG = (
    prepare_participation_ablation_experiment(
        experiment_version=(
            "ablation_a1_no_turns_keep_speaks"
        ),

        experiment_title=(
            "A1-T — No Filtered Turns, Keep Speaks"
        ),

        ablation_id=(
            "A1_NO_TURNS_KEEP_SPEAKS"
        ),

        include_speaks=True,

        include_filtered_turns=False,

        include_participation_assessment=True,
    )
)


A1 — No Participation Branch — CONFIGURATION READY
Ablation ID: A1_NO_PARTICIPATION_BRANCH
Include participant speaks: False
Include filtered turns: False
Include participation_assessment: False
Schema keys: ['local_temporal_assessment', 'global_temporal_assessment', 'temporal_assessment', 'semantic_assessment', 'decisive_dimension', 'label']
Full R1 prompt SHA256: 7da269dce7d27e966cf1ecfa16543f7c55d5180b93d95cb33c3d44460bb5483f
Ablation prompt SHA256: a60a8468b0fb22e3c94cbecfc2b5f2a83004a2b4749684193d993c8860c264ba
Prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_participation_branch/prompt_diff_vs_full_structured_r1.txt
Example prompt characters: 22483
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_participation_branch/predictions_cache.json
A1-S — No Speaks, Keep Filtered Turns — CONFIGURATION READY
Ablation ID: A1_NO_SPEAKS_KEEP_TURNS
Include participant speaks: False
Include filtered

# A1 — No Participation Branch

Removed from the model-facing input for both participants:

- `speaks`
- `filtered_turns`

Removed from the prompt:

- participation from `AVAILABLE EVIDENCE`
- the participation property from the operational definition
- the complete `PARTICIPATION VALIDITY` section
- raw-filtered-turn instructions and references
- participation-dependent bullets in the temporal/final decision sections
- empty Participant A/B blocks in `CURRENT CASE`

Removed from the structured output:

- `participation_assessment`
- `PARTICIPATION` as a possible `decisive_dimension`

Run the inspection cell first. It prints the exact rendered prompt and exact ablated payload for one real case.


In [ ]:
A1_NO_PARTICIPATION_INSPECTION = (
    inspect_participation_ablation_prompt(
        A1_NO_PARTICIPATION_CONFIG,
        case_index=0,
    )
)

PROMPT INSPECTION — A1 — No Participation Branch
Inspection case ID: consolidation_lag_2sec_000
Gold label is intentionally NOT printed inside the model prompt.

REMOVED / ADAPTED PROMPT COMPONENTS
{
  "participant_A_speaks": [
    "Speaks:",
    "{participant_A_speaks}"
  ],
  "participant_B_speaks": [
    "Speaks:",
    "{participant_B_speaks}"
  ],
  "participant_A_turns": [
    "Filtered turns:",
    "{participant_A_turns}"
  ],
  "participant_B_turns": [
    "Filtered turns:",
    "{participant_B_turns}"
  ],
  "available_evidence_adaptation": {
    "include_speaks": false,
    "include_filtered_turns": false
  },
  "participation_branch": [
    "PARTICIPATION VALIDITY section",
    "participation property in operational definition",
    "participation property in final decision",
    "participation-dependent cross-branch bullets",
    "participation_assessment output",
    "PARTICIPATION decisive_dimension option"
  ],
  "filtered_turn_instruction_removals": [
    "TURN FORMAT AN

In [ ]:
# Set print_each_case_prompt=True only if you want all 400 prompts printed.
A1_NO_PARTICIPATION_CACHE = (
    run_participation_ablation_experiment(
        A1_NO_PARTICIPATION_CONFIG,
        print_each_case_prompt=False,
    )
)

Created cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_participation_branch/predictions_cache.json


ablation_a1_no_participation_branch:   0%|          | 0/400 [00:00<?, ?it/s]


A1 — No Participation Branch — INFERENCE COMPLETE
Cached records: 400
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_participation_branch/predictions_cache.json


In [ ]:
A1_NO_PARTICIPATION_EVALUATION = (
    evaluate_reasoning_experiment(
        A1_NO_PARTICIPATION_CONFIG
    )
)

A1 — No Participation Branch — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.7575
Balanced accuracy: 0.6617
ANOMALOUS precision: 0.8285
ANOMALOUS recall: 0.8533
ANOMALOUS F1: 0.8407
NORMAL recall / specificity: 0.4700
MCC: 0.3340
Matched source-group exact rate: 0.2500
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,47,53
Gold ANOMALOUS,44,256



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.516484,0.470000,0.492147,100.0000
ANOMALOUS,0.828479,0.853333,0.840722,300.0000
accuracy,0.757500,0.757500,0.757500,0.7575
macro avg,0.672481,0.661667,0.666435,400.0000
weighted avg,0.750480,0.757500,0.753579,400.0000



PER CASE FAMILY


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,4,46,46,0.92,0.92,1.0
1,lag_3sec,50,50,0,3,47,47,0.94,0.94,1.0
2,normal,100,100,0,47,53,47,0.47,0.47,1.0
3,silent_partner,100,100,0,31,69,69,0.69,0.69,1.0
4,wrong_partner,100,100,0,6,94,94,0.94,0.94,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,MISSING,400
1,local_temporal_assessment,ANOMALOUS,186
2,local_temporal_assessment,LIMITED,118
3,local_temporal_assessment,NORMAL,96
4,global_temporal_assessment,ANOMALOUS,226
5,global_temporal_assessment,LIMITED,118
6,global_temporal_assessment,NORMAL,56
7,temporal_assessment,ANOMALOUS,226
8,temporal_assessment,LIMITED,118
9,temporal_assessment,NORMAL,56



Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_participation_branch/predictions_cache.json
Saved prompt: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_participation_branch/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_participation_branch/prompt_diff_vs_full_structured_r1.txt
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_participation_branch/predictions_all_400.csv
Saved reasoning assessments: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_participation_branch/reasoning_assessments.csv
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_participation_branch/classification_errors.csv

All 400 cases produced valid binary predictions.


In [ ]:
import pandas as pd
from IPython.display import display


# ============================================================
# LOAD A1 — NO PARTICIPATION BRANCH CASE-LEVEL RESULTS
# Run this after the A1 evaluation cell.
# ============================================================

if "A1_NO_PARTICIPATION_EVALUATION" in globals():

    a1_df = (
        A1_NO_PARTICIPATION_EVALUATION[
            "results_df"
        ].copy()
    )

elif "A1_NO_PARTICIPATION_CONFIG" in globals():

    a1_df = pd.read_csv(
        A1_NO_PARTICIPATION_CONFIG[
            "paths"
        ][
            "predictions_csv"
        ]
    )

else:

    raise RuntimeError(
        "Run the A1 No Participation configuration "
        "and evaluation cells first."
    )


# ============================================================
# NORMALIZE TEXT FIELDS
# ============================================================

upper_columns = [
    "gold_label",
    "prediction",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


for column in upper_columns:

    if column in a1_df.columns:

        a1_df[column] = (
            a1_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )


for column in [
    "case_family",
    "case_variant",
]:

    if column in a1_df.columns:

        a1_df[column] = (
            a1_df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


# ============================================================
# VALIDITY AND CORRECTNESS
# ============================================================

if "valid_prediction" not in a1_df.columns:

    a1_df["valid_prediction"] = (
        a1_df["prediction"].isin([
            "NORMAL",
            "ANOMALOUS",
        ])
    )


if "correct" not in a1_df.columns:

    a1_df["correct"] = (
        a1_df["valid_prediction"]
        &
        (
            a1_df["gold_label"]
            ==
            a1_df["prediction"]
        )
    )


# ============================================================
# EXPECTED VALUES FOR A1 STRUCTURED OUTPUT
# No participation_assessment exists in this experiment.
# ============================================================

A1_ASSESSMENT_LEVELS = {

    "local_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "global_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "semantic_assessment": [
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    ],

    "decisive_dimension": [
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ],
}


# ============================================================
# FREQUENCY TABLE
# ============================================================

def a1_frequency_table(
    subset,
    field,
    expected_values=None,
):

    if field not in subset.columns:

        return pd.DataFrame({
            "assessment_field": [field],
            "assessment_value": ["COLUMN_NOT_AVAILABLE"],
            "count": [0],
            "percentage": [0.0],
        })


    values = (
        subset[field]
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
    )


    if expected_values is None:

        expected_values = sorted(
            values.unique().tolist()
        )


    ordered_values = list(
        dict.fromkeys(
            list(expected_values)
            +
            ["MISSING"]
        )
    )


    # Also retain unexpected model outputs, if any.
    unexpected_values = [
        value
        for value in values.unique().tolist()
        if value not in ordered_values
    ]

    ordered_values.extend(
        sorted(unexpected_values)
    )


    counts = (
        values
        .value_counts(dropna=False)
        .reindex(
            ordered_values,
            fill_value=0,
        )
    )


    denominator = len(subset)


    table = pd.DataFrame({

        "assessment_field": field,

        "assessment_value": counts.index,

        "count": counts.values,
    })


    if denominator > 0:

        table["percentage"] = (
            100.0
            *
            table["count"]
            /
            denominator
        ).round(2)

    else:

        table["percentage"] = 0.0


    return table


# ============================================================
# SAFE CROSSTAB
# ============================================================

def display_a1_crosstab(
    subset,
    row_field,
    column_field,
    title,
):

    print(
        f"\n{title}"
    )


    if (
        row_field not in subset.columns
        or column_field not in subset.columns
    ):

        print(
            "One or both required columns are unavailable."
        )

        return


    display(
        pd.crosstab(
            subset[row_field].fillna("MISSING"),
            subset[column_field].fillna("MISSING"),
            margins=True,
        )
    )


# ============================================================
# COMPLETE SUBSET INSPECTION
# ============================================================

def inspect_a1_subset(
    subset,
    title,
):

    subset = subset.copy()


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        title
    )

    print(
        "=" * 100
    )


    if subset.empty:

        print(
            "No cases found in this subset."
        )

        return subset


    overview = pd.DataFrame([{

        "cases": len(subset),

        "gold_NORMAL": int(
            (
                subset["gold_label"]
                ==
                "NORMAL"
            ).sum()
        ),

        "gold_ANOMALOUS": int(
            (
                subset["gold_label"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "pred_NORMAL": int(
            (
                subset["prediction"]
                ==
                "NORMAL"
            ).sum()
        ),

        "pred_ANOMALOUS": int(
            (
                subset["prediction"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "invalid_predictions": int(
            (
                ~subset["valid_prediction"]
            ).sum()
        ),

        "correct": int(
            subset["correct"].sum()
        ),

        "accuracy_percent": round(
            100.0
            *
            subset["correct"].mean(),
            2,
        ),
    }])


    print(
        "\nSUBSET OVERVIEW"
    )

    display(
        overview
    )


    if "case_variant" in subset.columns:

        print(
            "\nCASE VARIANT BREAKDOWN"
        )


        variant_counts = (
            subset["case_variant"]
            .fillna("MISSING")
            .value_counts()
            .rename_axis("case_variant")
            .reset_index(name="count")
        )


        variant_counts["percentage"] = (
            100.0
            *
            variant_counts["count"]
            /
            len(subset)
        ).round(2)


        display(
            variant_counts
        )


    print(
        "\nASSESSMENT BREAKDOWN"
    )


    assessment_tables = []


    for (
        field,
        expected_values,
    ) in A1_ASSESSMENT_LEVELS.items():

        assessment_tables.append(
            a1_frequency_table(
                subset=subset,
                field=field,
                expected_values=expected_values,
            )
        )


    assessment_table = pd.concat(
        assessment_tables,
        ignore_index=True,
    )


    display(
        assessment_table
    )


    display_a1_crosstab(
        subset=subset,
        row_field="local_temporal_assessment",
        column_field="global_temporal_assessment",
        title=(
            "LOCAL × GLOBAL TEMPORAL ASSESSMENT"
        ),
    )


    display_a1_crosstab(
        subset=subset,
        row_field="temporal_assessment",
        column_field="decisive_dimension",
        title=(
            "COMBINED TEMPORAL × DECISIVE DIMENSION"
        ),
    )


    display_a1_crosstab(
        subset=subset,
        row_field="semantic_assessment",
        column_field="decisive_dimension",
        title=(
            "SEMANTIC ASSESSMENT × DECISIVE DIMENSION"
        ),
    )


    display_a1_crosstab(
        subset=subset,
        row_field="temporal_assessment",
        column_field="semantic_assessment",
        title=(
            "TEMPORAL × SEMANTIC ASSESSMENT"
        ),
    )


    return subset


# ============================================================
# CASE-FAMILY MATCHING
# Supports small naming differences in case_family.
# ============================================================

def get_a1_family_mask(
    dataframe,
    family,
):

    family_values = (
        dataframe["case_family"]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    if family == "normal":

        return (
            family_values.eq("normal")
            |
            family_values.eq("normals")
        )


    if family == "lag":

        return (
            family_values.eq("lag")
            |
            family_values.eq("lags")
            |
            family_values.str.contains(
                r"(?:^|_)lag(?:$|_)",
                regex=True,
            )
        )


    if family == "wrong_partner":

        return (
            family_values.eq("wrong_partner")
            |
            family_values.eq("wrong_partners")
            |
            (
                family_values.str.contains(
                    "wrong",
                    regex=False,
                )
                &
                family_values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    if family == "silent_partner":

        return (
            family_values.eq("silent_partner")
            |
            family_values.eq("silent_partners")
            |
            (
                family_values.str.contains(
                    "silent",
                    regex=False,
                )
                &
                family_values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    raise ValueError(
        f"Unknown family: {family}"
    )


# ============================================================
# CREATE ONE OUTCOME SUBSET
# ============================================================

def get_a1_outcome_subset(
    family,
    outcome,
):

    family_mask = get_a1_family_mask(
        dataframe=a1_df,
        family=family,
    )


    if family == "normal":

        gold_label = "NORMAL"

        if outcome == "correct":

            prediction = "NORMAL"

        elif outcome == "missed":

            prediction = "ANOMALOUS"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )

    else:

        gold_label = "ANOMALOUS"

        if outcome == "correct":

            prediction = "ANOMALOUS"

        elif outcome == "missed":

            prediction = "NORMAL"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )


    subset = a1_df[
        family_mask
        &
        (
            a1_df["gold_label"]
            ==
            gold_label
        )
        &
        (
            a1_df["prediction"]
            ==
            prediction
        )
    ].copy()


    return subset


# ============================================================
# INITIAL SANITY CHECK
# ============================================================

print(
    "A1 cases loaded:",
    len(a1_df),
)


print(
    "\nDETECTED CASE FAMILIES"
)

display(
    a1_df[
        "case_family"
    ]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("case_family")
    .reset_index(name="count")
)


print(
    "\nCASE FAMILY × PREDICTION"
)

display(
    pd.crosstab(
        a1_df["case_family"],
        a1_df["prediction"],
        margins=True,
    )
)

A1 cases loaded: 400

DETECTED CASE FAMILIES


,case_family,count
0,normal,100
1,wrong_partner,100
2,lag,100
3,silent_partner,100



CASE FAMILY × PREDICTION


prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,93,7,100
normal,53,47,100
silent_partner,69,31,100
wrong_partner,94,6,100
All,309,91,400


In [ ]:
# ============================================================
# 1. LAG — CORRECTLY DETECTED
# ============================================================

a1_lag_correct = get_a1_outcome_subset(
    family="lag",
    outcome="correct",
)

inspect_a1_subset(
    a1_lag_correct,
    (
        "A1 — LAG CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# 2. LAG — NOT DETECTED
# ============================================================

a1_lag_missed = get_a1_outcome_subset(
    family="lag",
    outcome="missed",
)

inspect_a1_subset(
    a1_lag_missed,
    (
        "A1 — LAG CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# 3. WRONG PARTNER — CORRECTLY DETECTED
# ============================================================

a1_wrong_partner_correct = get_a1_outcome_subset(
    family="wrong_partner",
    outcome="correct",
)

inspect_a1_subset(
    a1_wrong_partner_correct,
    (
        "A1 — WRONG-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# 4. WRONG PARTNER — NOT DETECTED
# ============================================================

a1_wrong_partner_missed = get_a1_outcome_subset(
    family="wrong_partner",
    outcome="missed",
)

inspect_a1_subset(
    a1_wrong_partner_missed,
    (
        "A1 — WRONG-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# 5. NORMAL — CORRECTLY DETECTED
# ============================================================

a1_normal_correct = get_a1_outcome_subset(
    family="normal",
    outcome="correct",
)

inspect_a1_subset(
    a1_normal_correct,
    (
        "A1 — NORMAL CASES CORRECTLY "
        "PREDICTED AS NORMAL"
    ),
)


# ============================================================
# 6. NORMAL — NOT RECOGNIZED
# ============================================================

a1_normal_missed = get_a1_outcome_subset(
    family="normal",
    outcome="missed",
)

inspect_a1_subset(
    a1_normal_missed,
    (
        "A1 — NORMAL CASES INCORRECTLY "
        "PREDICTED AS ANOMALOUS"
    ),
)


# ============================================================
# 7. SILENT PARTNER — CORRECTLY DETECTED
# ============================================================

a1_silent_partner_correct = get_a1_outcome_subset(
    family="silent_partner",
    outcome="correct",
)

inspect_a1_subset(
    a1_silent_partner_correct,
    (
        "A1 — SILENT-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# 8. SILENT PARTNER — NOT DETECTED
# ============================================================

a1_silent_partner_missed = get_a1_outcome_subset(
    family="silent_partner",
    outcome="missed",
)

inspect_a1_subset(
    a1_silent_partner_missed,
    (
        "A1 — SILENT-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


A1 — LAG CASES CORRECTLY DETECTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,invalid_predictions,correct,accuracy_percent
0,93,0,93,0,93,0,93,100.0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_3sec,47,50.54
1,lag_2sec,46,49.46



ASSESSMENT BREAKDOWN


...


SEMANTIC ASSESSMENT × DECISIVE DIMENSION


decisive_dimension,SEMANTIC,All
semantic_assessment,,
COMPATIBLE,31,31
All,31,31



TEMPORAL × SEMANTIC ASSESSMENT


semantic_assessment,COMPATIBLE,All
temporal_assessment,,
LIMITED,31,31
All,31,31


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency
305,consolidation_silent_partner_005,heldout_source_005,silent_partner,silent_partner,ANOMALOUS,NORMAL,None,LIMITED,LIMITED,LIMITED,...,4350,8.8328,"```json\n{\n ""local_temporal_assessment"": ""LI...",None,True,False,False,False,False,False
306,consolidation_silent_partner_006,heldout_source_006,silent_partner,silent_partner,ANOMALOUS,NORMAL,None,LIMITED,LIMITED,LIMITED,...,4389,8.8468,"```json\n{\n ""local_temporal_assessment"": ""LI...",None,True,False,False,False,False,False
309,consolidation_silent_partner_009,heldout_source_009,silent_partner,silent_partner,ANOMALOUS,NORMAL,None,LIMITED,LIMITED,LIMITED,...,4257,8.7202,"```json\n{\n ""local_temporal_assessment"": ""LI...",None,True,False,False,False,False,False
314,consolidation_silent_partner_014,heldout_source_014,silent_partner,silent_partner,ANOMALOUS,NORMAL,None,LIMITED,LIMITED,LIMITED,...,4326,8.6659,"```json\n{\n ""local_temporal_assessment"": ""LI...",None,True,False,False,False,False,False
315,consolidation_silent_partner_015,heldout_source_015,silent_partner,silent_partner,ANOMALOUS,NORMAL,None,LIMITED,LIMITED,LIMITED,...,4289,8.7483,"```json\n{\n ""local_temporal_assessment"": ""LI...",None,True,False,False,False,False,False
325,consolidation_silent_partner_025,heldout_source_025,silent_partner,silent_partner,ANOMALOUS,NORMAL,None,LIMITED,LIMITED,LIMITED,...,4349,8.6819,"```json\n{\n ""local_temporal_assessment"": ""LI...",None,True,False,False,False,False,False
327,consolidation_silent_partner_027,heldout_source_027,silent_partner,silent_partner,ANOMALOUS,NORMAL,None,LIMITED,LIMITED,LIMITED,...,4317,8.6755,"```json\n{\n ""local_temporal_assessment"": ""LI...",None,True,False,False,False,False,False
331,consolidation_silent_partner_031,heldout_source_031,silent_partner,silent_partner,ANOMALOUS,NORMAL,None,LIMITED,LIMITED,LIMITED,...,4285,8.7255,"```json\n{\n ""local_temporal_assessment"": ""LI...",None,True,False,False,False,False,False
332,consolidation_silent_partner_032,heldout_source_032,silent_partner,silent_partner,ANOMALOUS,NORMAL,None,LIMITED,LIMITED,LIMITED,...,4466,8.7656,"```json\n{\n ""local_temporal_assessment"": ""LI...",None,True,False,False,False,False,False
336,consolidation_silent_partner_036,heldout_source_036,silent_partner,silent_partner,ANOMALOUS,NORMAL,None,LIMITED,LIMITED,LIMITED,...,4395,8.7842,"```json\n{\n ""local_temporal_assessment"": ""LI...",None,True,False,False,False,False,False


# A1-S — Remove `speaks`, Keep Filtered Turns

Removed from the model-facing input for both participants:

- `speaks`

Prompt adaptation:

- removes `speaks` from `AVAILABLE EVIDENCE`
- removes the `speaks` field definition
- keeps the participation branch and evaluates it using only the actual filtered turn lists

Retained exactly as in Full Structured R1:

- `filtered_turns`
- `participation_assessment`
- every temporal and semantic branch
- the full Structured R1 output schema


In [ ]:
A1_NO_SPEAKS_INSPECTION = (
    inspect_participation_ablation_prompt(
        A1_NO_SPEAKS_CONFIG,
        case_index=0,
    )
)

PROMPT INSPECTION — A1-S — No Speaks, Keep Filtered Turns
Inspection case ID: consolidation_lag_2sec_000
Gold label is intentionally NOT printed inside the model prompt.

REMOVED / ADAPTED PROMPT COMPONENTS
{
  "participant_A_speaks": [
    "Speaks:",
    "{participant_A_speaks}"
  ],
  "participant_B_speaks": [
    "Speaks:",
    "{participant_B_speaks}"
  ],
  "available_evidence_adaptation": {
    "include_speaks": false,
    "include_filtered_turns": true
  },
  "participation_instruction_adaptation": [
    "Removed speaks-field definition",
    "Participation assessment now uses filtered turns only"
  ]
}

EXACT ABLATED MODEL INPUT PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "filtered_turns": [
      [
        9.89,
        10.78
      ],
      [
        11.65,
        18.24
      ],
      [
        25.38,
        27.71
      ],
      [
        32.83,
        33.76
      ],
      [
        35.65,
        44.83
      ],
      [
        68.67,
        75

In [ ]:
A1_NO_SPEAKS_CACHE = (
    run_participation_ablation_experiment(
        A1_NO_SPEAKS_CONFIG,
        print_each_case_prompt=False,
    )
)

Created cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_speaks_keep_turns/predictions_cache.json


ablation_a1_no_speaks_keep_turns:   0%|          | 0/400 [00:00<?, ?it/s]


A1-S — No Speaks, Keep Filtered Turns — INFERENCE COMPLETE
Cached records: 400
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_speaks_keep_turns/predictions_cache.json


In [ ]:
A1_NO_SPEAKS_EVALUATION = (
    evaluate_reasoning_experiment(
        A1_NO_SPEAKS_CONFIG
    )
)

A1-S — No Speaks, Keep Filtered Turns — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.8775
Balanced accuracy: 0.8350
ANOMALOUS precision: 0.9169
ANOMALOUS recall: 0.9200
ANOMALOUS F1: 0.9185
NORMAL recall / specificity: 0.7500
MCC: 0.6723
Matched source-group exact rate: 0.5800
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,75,25
Gold ANOMALOUS,24,276



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.757576,0.7500,0.753769,100.0000
ANOMALOUS,0.916944,0.9200,0.918469,300.0000
accuracy,0.877500,0.8775,0.877500,0.8775
macro avg,0.837260,0.8350,0.836119,400.0000
weighted avg,0.877102,0.8775,0.877294,400.0000



PER CASE FAMILY


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,12,38,38,0.76,0.76,1.0
1,lag_3sec,50,50,0,6,44,44,0.88,0.88,1.0
2,normal,100,100,0,75,25,75,0.75,0.75,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,6,94,94,0.94,0.94,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,VALID,300
1,participation_assessment,INVALID,100
2,local_temporal_assessment,NORMAL,164
3,local_temporal_assessment,ANOMALOUS,159
4,local_temporal_assessment,LIMITED,77
5,global_temporal_assessment,ANOMALOUS,217
6,global_temporal_assessment,NORMAL,106
7,global_temporal_assessment,LIMITED,77
8,temporal_assessment,ANOMALOUS,217
9,temporal_assessment,NORMAL,106



Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_speaks_keep_turns/predictions_cache.json
Saved prompt: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_speaks_keep_turns/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_speaks_keep_turns/prompt_diff_vs_full_structured_r1.txt
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_speaks_keep_turns/predictions_all_400.csv
Saved reasoning assessments: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_speaks_keep_turns/reasoning_assessments.csv
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_speaks_keep_turns/classification_errors.csv

All 400 cases produced valid binary predictions.


In [ ]:
import pandas as pd
from IPython.display import display


# ============================================================
# LOAD A1-S — NO SPEAKS, KEEP FILTERED TURNS
# ============================================================

if "A1_NO_SPEAKS_EVALUATION" in globals():

    a1s_df = (
        A1_NO_SPEAKS_EVALUATION[
            "results_df"
        ].copy()
    )

elif "A1_NO_SPEAKS_CONFIG" in globals():

    a1s_df = pd.read_csv(
        A1_NO_SPEAKS_CONFIG[
            "paths"
        ][
            "predictions_csv"
        ]
    )

else:

    raise RuntimeError(
        "Run the A1-S configuration and evaluation cells first."
    )


# ============================================================
# NORMALIZE TEXT FIELDS
# ============================================================

upper_columns = [
    "gold_label",
    "prediction",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


for column in upper_columns:

    if column in a1s_df.columns:

        a1s_df[column] = (
            a1s_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )


for column in [
    "case_family",
    "case_variant",
]:

    if column in a1s_df.columns:

        a1s_df[column] = (
            a1s_df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


# ============================================================
# VALID PREDICTION AND CORRECTNESS
# ============================================================

if "valid_prediction" not in a1s_df.columns:

    a1s_df["valid_prediction"] = (
        a1s_df["prediction"].isin([
            "NORMAL",
            "ANOMALOUS",
        ])
    )


if "correct" not in a1s_df.columns:

    a1s_df["correct"] = (
        a1s_df["valid_prediction"]
        &
        (
            a1s_df["gold_label"]
            ==
            a1s_df["prediction"]
        )
    )


# ============================================================
# EXPECTED STRUCTURED OUTPUT VALUES
# ============================================================

A1S_ASSESSMENT_LEVELS = {

    "participation_assessment": [
        "VALID",
        "INVALID",
    ],

    "local_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "global_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "semantic_assessment": [
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    ],

    "decisive_dimension": [
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ],
}


# ============================================================
# FREQUENCY TABLE
# ============================================================

def a1s_frequency_table(
    subset,
    field,
    expected_values,
):

    if field not in subset.columns:

        return pd.DataFrame({
            "assessment_field": [field],
            "assessment_value": ["COLUMN_NOT_AVAILABLE"],
            "count": [0],
            "percentage": [0.0],
        })


    values = (
        subset[field]
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
    )


    ordered_values = list(
        dict.fromkeys(
            list(expected_values)
            +
            ["MISSING"]
        )
    )


    unexpected_values = [
        value
        for value in values.unique().tolist()
        if value not in ordered_values
    ]


    ordered_values.extend(
        sorted(unexpected_values)
    )


    counts = (
        values
        .value_counts(dropna=False)
        .reindex(
            ordered_values,
            fill_value=0,
        )
    )


    table = pd.DataFrame({
        "assessment_field": field,
        "assessment_value": counts.index,
        "count": counts.values,
    })


    if len(subset) > 0:

        table["percentage"] = (
            100.0
            *
            table["count"]
            /
            len(subset)
        ).round(2)

    else:

        table["percentage"] = 0.0


    return table


# ============================================================
# SAFE CROSSTAB
# ============================================================

def display_a1s_crosstab(
    subset,
    row_field,
    column_field,
    title,
):

    print(
        f"\n{title}"
    )


    if (
        row_field not in subset.columns
        or
        column_field not in subset.columns
    ):

        print(
            "Required columns are unavailable."
        )

        return


    display(
        pd.crosstab(
            subset[row_field].fillna("MISSING"),
            subset[column_field].fillna("MISSING"),
            margins=True,
        )
    )


# ============================================================
# COMPLETE SUBSET INSPECTION
# ============================================================

def inspect_a1s_subset(
    subset,
    title,
):

    subset = subset.copy()


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        title
    )

    print(
        "=" * 100
    )


    if subset.empty:

        print(
            "No cases found in this subset."
        )

        return subset


    overview = pd.DataFrame([{

        "cases": len(subset),

        "gold_NORMAL": int(
            (
                subset["gold_label"]
                ==
                "NORMAL"
            ).sum()
        ),

        "gold_ANOMALOUS": int(
            (
                subset["gold_label"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "pred_NORMAL": int(
            (
                subset["prediction"]
                ==
                "NORMAL"
            ).sum()
        ),

        "pred_ANOMALOUS": int(
            (
                subset["prediction"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "invalid_predictions": int(
            (
                ~subset["valid_prediction"]
            ).sum()
        ),

        "correct": int(
            subset["correct"].sum()
        ),

        "accuracy_percent": round(
            100.0
            *
            subset["correct"].mean(),
            2,
        ),
    }])


    print(
        "\nSUBSET OVERVIEW"
    )

    display(
        overview
    )


    print(
        "\nCASE VARIANT BREAKDOWN"
    )


    variant_counts = (
        subset["case_variant"]
        .fillna("MISSING")
        .value_counts()
        .rename_axis("case_variant")
        .reset_index(name="count")
    )


    variant_counts["percentage"] = (
        100.0
        *
        variant_counts["count"]
        /
        len(subset)
    ).round(2)


    display(
        variant_counts
    )


    print(
        "\nASSESSMENT BREAKDOWN"
    )


    assessment_table = pd.concat(
        [
            a1s_frequency_table(
                subset=subset,
                field=field,
                expected_values=expected_values,
            )
            for field, expected_values
            in A1S_ASSESSMENT_LEVELS.items()
        ],
        ignore_index=True,
    )


    display(
        assessment_table
    )


    display_a1s_crosstab(
        subset,
        "participation_assessment",
        "decisive_dimension",
        (
            "PARTICIPATION ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_a1s_crosstab(
        subset,
        "local_temporal_assessment",
        "global_temporal_assessment",
        (
            "LOCAL × GLOBAL "
            "TEMPORAL ASSESSMENT"
        ),
    )


    display_a1s_crosstab(
        subset,
        "temporal_assessment",
        "decisive_dimension",
        (
            "COMBINED TEMPORAL "
            "× DECISIVE DIMENSION"
        ),
    )


    display_a1s_crosstab(
        subset,
        "semantic_assessment",
        "decisive_dimension",
        (
            "SEMANTIC ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_a1s_crosstab(
        subset,
        "temporal_assessment",
        "semantic_assessment",
        (
            "TEMPORAL × SEMANTIC ASSESSMENT"
        ),
    )


    return subset


# ============================================================
# FAMILY MASK
# ============================================================

def get_a1s_family_mask(
    dataframe,
    family,
):

    values = (
        dataframe["case_family"]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    if family == "normal":

        return values.isin([
            "normal",
            "normals",
        ])


    if family == "lag":

        return (
            values.isin([
                "lag",
                "lags",
            ])
            |
            values.str.contains(
                r"(?:^|_)lag(?:$|_)",
                regex=True,
            )
        )


    if family == "wrong_partner":

        return (
            values.isin([
                "wrong_partner",
                "wrong_partners",
            ])
            |
            (
                values.str.contains(
                    "wrong",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    if family == "silent_partner":

        return (
            values.isin([
                "silent_partner",
                "silent_partners",
            ])
            |
            (
                values.str.contains(
                    "silent",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    raise ValueError(
        f"Unknown family: {family}"
    )


# ============================================================
# CREATE OUTCOME SUBSET
# ============================================================

def get_a1s_outcome_subset(
    family,
    outcome,
):

    family_mask = get_a1s_family_mask(
        dataframe=a1s_df,
        family=family,
    )


    if family == "normal":

        gold_label = "NORMAL"

        if outcome == "correct":

            prediction = "NORMAL"

        elif outcome == "missed":

            prediction = "ANOMALOUS"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )

    else:

        gold_label = "ANOMALOUS"

        if outcome == "correct":

            prediction = "ANOMALOUS"

        elif outcome == "missed":

            prediction = "NORMAL"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )


    return a1s_df[
        family_mask
        &
        (
            a1s_df["gold_label"]
            ==
            gold_label
        )
        &
        (
            a1s_df["prediction"]
            ==
            prediction
        )
    ].copy()


# ============================================================
# INITIAL CHECK
# ============================================================

print(
    "A1-S cases loaded:",
    len(a1s_df),
)


print(
    "\nCASE FAMILY COUNTS"
)

display(
    a1s_df["case_family"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("case_family")
    .reset_index(name="count")
)


print(
    "\nCASE FAMILY × PREDICTION"
)

display(
    pd.crosstab(
        a1s_df["case_family"],
        a1s_df["prediction"],
        margins=True,
    )
)

A1-S cases loaded: 400

CASE FAMILY COUNTS


,case_family,count
0,normal,100
1,wrong_partner,100
2,lag,100
3,silent_partner,100



CASE FAMILY × PREDICTION


prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,82,18,100
normal,25,75,100
silent_partner,100,0,100
wrong_partner,94,6,100
All,301,99,400


In [ ]:
# ============================================================
# LAG — CORRECTLY DETECTED
# ============================================================

a1s_lag_correct = get_a1s_outcome_subset(
    family="lag",
    outcome="correct",
)

inspect_a1s_subset(
    a1s_lag_correct,
    (
        "A1-S — LAG CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# LAG — MISSED
# ============================================================

a1s_lag_missed = get_a1s_outcome_subset(
    family="lag",
    outcome="missed",
)

inspect_a1s_subset(
    a1s_lag_missed,
    (
        "A1-S — LAG CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# WRONG PARTNER — CORRECTLY DETECTED
# ============================================================

a1s_wrong_partner_correct = get_a1s_outcome_subset(
    family="wrong_partner",
    outcome="correct",
)

inspect_a1s_subset(
    a1s_wrong_partner_correct,
    (
        "A1-S — WRONG-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# WRONG PARTNER — MISSED
# ============================================================

a1s_wrong_partner_missed = get_a1s_outcome_subset(
    family="wrong_partner",
    outcome="missed",
)

inspect_a1s_subset(
    a1s_wrong_partner_missed,
    (
        "A1-S — WRONG-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# NORMAL — CORRECTLY DETECTED
# ============================================================

a1s_normal_correct = get_a1s_outcome_subset(
    family="normal",
    outcome="correct",
)

inspect_a1s_subset(
    a1s_normal_correct,
    (
        "A1-S — NORMAL CASES CORRECTLY "
        "PREDICTED AS NORMAL"
    ),
)


# ============================================================
# NORMAL — FALSELY FLAGGED
# ============================================================

a1s_normal_missed = get_a1s_outcome_subset(
    family="normal",
    outcome="missed",
)

inspect_a1s_subset(
    a1s_normal_missed,
    (
        "A1-S — NORMAL CASES INCORRECTLY "
        "PREDICTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — CORRECTLY DETECTED
# ============================================================

a1s_silent_partner_correct = get_a1s_outcome_subset(
    family="silent_partner",
    outcome="correct",
)

inspect_a1s_subset(
    a1s_silent_partner_correct,
    (
        "A1-S — SILENT-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — MISSED
# ============================================================

a1s_silent_partner_missed = get_a1s_outcome_subset(
    family="silent_partner",
    outcome="missed",
)

inspect_a1s_subset(
    a1s_silent_partner_missed,
    (
        "A1-S — SILENT-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


A1-S — LAG CASES CORRECTLY DETECTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,invalid_predictions,correct,accuracy_percent
0,82,0,82,0,82,0,82,100.0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_3sec,44,53.66
1,lag_2sec,38,46.34



ASSESSMENT BREAKDOWN


...

decisive_dimension,PARTICIPATION,SEMANTIC,TEMPORAL,All
semantic_assessment,,,,
COMPATIBLE,8,0,25,33
INCOMPATIBLE,0,3,0,3
LIMITED,64,0,0,64
All,72,3,25,100



TEMPORAL × SEMANTIC ASSESSMENT


semantic_assessment,COMPATIBLE,INCOMPATIBLE,LIMITED,All
temporal_assessment,,,,
ANOMALOUS,25,3,21,49
LIMITED,8,0,43,51
All,33,3,64,100



A1-S — SILENT-PARTNER CASES NOT DETECTED (PREDICTED NORMAL)
No cases found in this subset.


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency


# A1-T — Keep `speaks`, Remove Filtered Turns

Removed from the model-facing input for both participants:

- `filtered_turns`

Prompt adaptation:

- removes filtered turns from `AVAILABLE EVIDENCE`
- removes the complete raw-turn format/backchannel section
- removes other instructions that require inspection of raw turn lists
- keeps the participation branch and evaluates it using only `speaks`

Retained exactly as in Full Structured R1:

- `speaks`
- `participation_assessment`
- every local/global temporal feature
- every semantic branch
- the full Structured R1 output schema


In [ ]:
A1_NO_TURNS_INSPECTION = (
    inspect_participation_ablation_prompt(
        A1_NO_TURNS_CONFIG,
        case_index=0,
    )
)

PROMPT INSPECTION — A1-T — No Filtered Turns, Keep Speaks
Inspection case ID: consolidation_lag_2sec_000
Gold label is intentionally NOT printed inside the model prompt.

REMOVED / ADAPTED PROMPT COMPONENTS
{
  "participant_A_turns": [
    "Filtered turns:",
    "{participant_A_turns}"
  ],
  "participant_B_turns": [
    "Filtered turns:",
    "{participant_B_turns}"
  ],
  "available_evidence_adaptation": {
    "include_speaks": true,
    "include_filtered_turns": false
  },
  "participation_instruction_adaptation": [
    "Participation assessment now uses speaks fields only"
  ],
  "filtered_turn_instruction_removals": [
    "TURN FORMAT AND BACKCHANNEL FILTERING section",
    "raw-turn-structure local-reference bullet",
    "raw-turn-structure overlap bullet",
    "observed Participant B turns statement"
  ]
}

EXACT ABLATED MODEL INPUT PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true
  },
  "participant_B": {
    "speaks": true
  },
  "local_t

In [ ]:
A1_NO_TURNS_CACHE = (
    run_participation_ablation_experiment(
        A1_NO_TURNS_CONFIG,
        print_each_case_prompt=False,
    )
)

Created cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_turns_keep_speaks/predictions_cache.json


ablation_a1_no_turns_keep_speaks:   0%|          | 0/400 [00:00<?, ?it/s]


A1-T — No Filtered Turns, Keep Speaks — INFERENCE COMPLETE
Cached records: 400
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_turns_keep_speaks/predictions_cache.json


In [ ]:
A1_NO_TURNS_EVALUATION = (
    evaluate_reasoning_experiment(
        A1_NO_TURNS_CONFIG
    )
)

A1-T — No Filtered Turns, Keep Speaks — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.8925
Balanced accuracy: 0.8550
ANOMALOUS precision: 0.9269
ANOMALOUS recall: 0.9300
ANOMALOUS F1: 0.9285
NORMAL recall / specificity: 0.7800
MCC: 0.7124
Matched source-group exact rate: 0.6400
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,78,22
Gold ANOMALOUS,21,279



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.787879,0.7800,0.783920,100.0000
ANOMALOUS,0.926910,0.9300,0.928453,300.0000
accuracy,0.892500,0.8925,0.892500,0.8925
macro avg,0.857395,0.8550,0.856186,400.0000
weighted avg,0.892152,0.8925,0.892319,400.0000



PER CASE FAMILY


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,6,44,44,0.88,0.88,1.0
1,lag_3sec,50,50,0,4,46,46,0.92,0.92,1.0
2,normal,100,100,0,78,22,78,0.78,0.78,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,11,89,89,0.89,0.89,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,VALID,300
1,participation_assessment,INVALID,100
2,local_temporal_assessment,NORMAL,207
3,local_temporal_assessment,LIMITED,120
4,local_temporal_assessment,ANOMALOUS,73
5,global_temporal_assessment,ANOMALOUS,168
6,global_temporal_assessment,LIMITED,120
7,global_temporal_assessment,NORMAL,112
8,temporal_assessment,ANOMALOUS,168
9,temporal_assessment,LIMITED,120



Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_turns_keep_speaks/predictions_cache.json
Saved prompt: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_turns_keep_speaks/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_turns_keep_speaks/prompt_diff_vs_full_structured_r1.txt
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_turns_keep_speaks/predictions_all_400.csv
Saved reasoning assessments: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_turns_keep_speaks/reasoning_assessments.csv
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/ablation_a1_no_turns_keep_speaks/classification_errors.csv

All 400 cases produced valid binary predictions.


In [ ]:
import pandas as pd
from IPython.display import display


# ============================================================
# LOAD A1-NT — NO FILTERED TURNS
# ============================================================

if "A1_NO_TURNS_EVALUATION" in globals():

    a1nt_df = (
        A1_NO_TURNS_EVALUATION[
            "results_df"
        ].copy()
    )

elif "A1_NO_TURNS_CONFIG" in globals():

    a1nt_df = pd.read_csv(
        A1_NO_TURNS_CONFIG[
            "paths"
        ][
            "predictions_csv"
        ]
    )

else:

    raise RuntimeError(
        "Run the A1_NO_TURNS configuration "
        "and evaluation cells first."
    )


# ============================================================
# NORMALIZE TEXT FIELDS
# ============================================================

upper_columns = [
    "gold_label",
    "prediction",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


for column in upper_columns:

    if column in a1nt_df.columns:

        a1nt_df[column] = (
            a1nt_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )


for column in [
    "case_family",
    "case_variant",
]:

    if column in a1nt_df.columns:

        a1nt_df[column] = (
            a1nt_df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


# ============================================================
# VALID PREDICTION AND CORRECTNESS
# ============================================================

if "valid_prediction" not in a1nt_df.columns:

    a1nt_df["valid_prediction"] = (
        a1nt_df["prediction"].isin([
            "NORMAL",
            "ANOMALOUS",
        ])
    )


if "correct" not in a1nt_df.columns:

    a1nt_df["correct"] = (
        a1nt_df["valid_prediction"]
        &
        (
            a1nt_df["gold_label"]
            ==
            a1nt_df["prediction"]
        )
    )


# ============================================================
# EXPECTED STRUCTURED OUTPUT VALUES
# ============================================================

A1NT_ASSESSMENT_LEVELS = {

    "participation_assessment": [
        "VALID",
        "INVALID",
    ],

    "local_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "global_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "semantic_assessment": [
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    ],

    "decisive_dimension": [
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ],
}


# ============================================================
# FREQUENCY TABLE
# ============================================================

def a1nt_frequency_table(
    subset,
    field,
    expected_values,
):

    if field not in subset.columns:

        return pd.DataFrame({
            "assessment_field": [field],
            "assessment_value": ["COLUMN_NOT_AVAILABLE"],
            "count": [0],
            "percentage": [0.0],
        })


    values = (
        subset[field]
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
    )


    ordered_values = list(
        dict.fromkeys(
            list(expected_values)
            +
            ["MISSING"]
        )
    )


    unexpected_values = [
        value
        for value in values.unique().tolist()
        if value not in ordered_values
    ]


    ordered_values.extend(
        sorted(unexpected_values)
    )


    counts = (
        values
        .value_counts(dropna=False)
        .reindex(
            ordered_values,
            fill_value=0,
        )
    )


    table = pd.DataFrame({
        "assessment_field": field,
        "assessment_value": counts.index,
        "count": counts.values,
    })


    if len(subset) > 0:

        table["percentage"] = (
            100.0
            *
            table["count"]
            /
            len(subset)
        ).round(2)

    else:

        table["percentage"] = 0.0


    return table


# ============================================================
# SAFE CROSSTAB
# ============================================================

def display_a1nt_crosstab(
    subset,
    row_field,
    column_field,
    title,
):

    print(
        f"\n{title}"
    )


    if (
        row_field not in subset.columns
        or
        column_field not in subset.columns
    ):

        print(
            "Required columns are unavailable."
        )

        return


    display(
        pd.crosstab(
            subset[row_field].fillna("MISSING"),
            subset[column_field].fillna("MISSING"),
            margins=True,
        )
    )


# ============================================================
# COMPLETE SUBSET INSPECTION
# ============================================================

def inspect_a1nt_subset(
    subset,
    title,
):

    subset = subset.copy()


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        title
    )

    print(
        "=" * 100
    )


    if subset.empty:

        print(
            "No cases found in this subset."
        )

        return subset


    overview = pd.DataFrame([{

        "cases": len(subset),

        "gold_NORMAL": int(
            (
                subset["gold_label"]
                ==
                "NORMAL"
            ).sum()
        ),

        "gold_ANOMALOUS": int(
            (
                subset["gold_label"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "pred_NORMAL": int(
            (
                subset["prediction"]
                ==
                "NORMAL"
            ).sum()
        ),

        "pred_ANOMALOUS": int(
            (
                subset["prediction"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "invalid_predictions": int(
            (
                ~subset["valid_prediction"]
            ).sum()
        ),

        "correct": int(
            subset["correct"].sum()
        ),

        "accuracy_percent": round(
            100.0
            *
            subset["correct"].mean(),
            2,
        ),
    }])


    print(
        "\nSUBSET OVERVIEW"
    )

    display(
        overview
    )


    print(
        "\nCASE VARIANT BREAKDOWN"
    )


    variant_counts = (
        subset["case_variant"]
        .fillna("MISSING")
        .value_counts()
        .rename_axis("case_variant")
        .reset_index(name="count")
    )


    variant_counts["percentage"] = (
        100.0
        *
        variant_counts["count"]
        /
        len(subset)
    ).round(2)


    display(
        variant_counts
    )


    print(
        "\nASSESSMENT BREAKDOWN"
    )


    assessment_table = pd.concat(
        [
            a1nt_frequency_table(
                subset=subset,
                field=field,
                expected_values=expected_values,
            )
            for field, expected_values
            in A1NT_ASSESSMENT_LEVELS.items()
        ],
        ignore_index=True,
    )


    display(
        assessment_table
    )


    display_a1nt_crosstab(
        subset,
        "participation_assessment",
        "decisive_dimension",
        (
            "PARTICIPATION ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_a1nt_crosstab(
        subset,
        "local_temporal_assessment",
        "global_temporal_assessment",
        (
            "LOCAL × GLOBAL "
            "TEMPORAL ASSESSMENT"
        ),
    )


    display_a1nt_crosstab(
        subset,
        "temporal_assessment",
        "decisive_dimension",
        (
            "COMBINED TEMPORAL "
            "× DECISIVE DIMENSION"
        ),
    )


    display_a1nt_crosstab(
        subset,
        "semantic_assessment",
        "decisive_dimension",
        (
            "SEMANTIC ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_a1nt_crosstab(
        subset,
        "temporal_assessment",
        "semantic_assessment",
        (
            "TEMPORAL × SEMANTIC ASSESSMENT"
        ),
    )


    return subset


# ============================================================
# FAMILY MASK
# ============================================================

def get_a1nt_family_mask(
    dataframe,
    family,
):

    values = (
        dataframe["case_family"]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    if family == "normal":

        return values.isin([
            "normal",
            "normals",
        ])


    if family == "lag":

        return (
            values.isin([
                "lag",
                "lags",
            ])
            |
            values.str.contains(
                r"(?:^|_)lag(?:$|_)",
                regex=True,
            )
        )


    if family == "wrong_partner":

        return (
            values.isin([
                "wrong_partner",
                "wrong_partners",
            ])
            |
            (
                values.str.contains(
                    "wrong",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    if family == "silent_partner":

        return (
            values.isin([
                "silent_partner",
                "silent_partners",
            ])
            |
            (
                values.str.contains(
                    "silent",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    raise ValueError(
        f"Unknown family: {family}"
    )


# ============================================================
# CREATE OUTCOME SUBSET
# ============================================================

def get_a1nt_outcome_subset(
    family,
    outcome,
):

    family_mask = get_a1nt_family_mask(
        dataframe=a1nt_df,
        family=family,
    )


    if family == "normal":

        gold_label = "NORMAL"

        if outcome == "correct":

            prediction = "NORMAL"

        elif outcome == "missed":

            prediction = "ANOMALOUS"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )

    else:

        gold_label = "ANOMALOUS"

        if outcome == "correct":

            prediction = "ANOMALOUS"

        elif outcome == "missed":

            prediction = "NORMAL"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )


    return a1nt_df[
        family_mask
        &
        (
            a1nt_df["gold_label"]
            ==
            gold_label
        )
        &
        (
            a1nt_df["prediction"]
            ==
            prediction
        )
    ].copy()


# ============================================================
# INITIAL CHECK
# ============================================================

print(
    "A1-NT cases loaded:",
    len(a1nt_df),
)


print(
    "\nCASE FAMILY COUNTS"
)

display(
    a1nt_df["case_family"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("case_family")
    .reset_index(name="count")
)


print(
    "\nCASE FAMILY × PREDICTION"
)

display(
    pd.crosstab(
        a1nt_df["case_family"],
        a1nt_df["prediction"],
        margins=True,
    )
)

A1-NT cases loaded: 400

CASE FAMILY COUNTS


,case_family,count
0,normal,100
1,wrong_partner,100
2,lag,100
3,silent_partner,100



CASE FAMILY × PREDICTION


prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,90,10,100
normal,22,78,100
silent_partner,100,0,100
wrong_partner,89,11,100
All,301,99,400


In [ ]:
# ============================================================
# LAG — CORRECTLY DETECTED
# ============================================================

a1nt_lag_correct = get_a1nt_outcome_subset(
    family="lag",
    outcome="correct",
)

inspect_a1nt_subset(
    a1nt_lag_correct,
    (
        "A1-NT — LAG CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# LAG — MISSED
# ============================================================

a1nt_lag_missed = get_a1nt_outcome_subset(
    family="lag",
    outcome="missed",
)

inspect_a1nt_subset(
    a1nt_lag_missed,
    (
        "A1-NT — LAG CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# WRONG PARTNER — CORRECTLY DETECTED
# ============================================================

a1nt_wrong_partner_correct = get_a1nt_outcome_subset(
    family="wrong_partner",
    outcome="correct",
)

inspect_a1nt_subset(
    a1nt_wrong_partner_correct,
    (
        "A1-NT — WRONG-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# WRONG PARTNER — MISSED
# ============================================================

a1nt_wrong_partner_missed = get_a1nt_outcome_subset(
    family="wrong_partner",
    outcome="missed",
)

inspect_a1nt_subset(
    a1nt_wrong_partner_missed,
    (
        "A1-NT — WRONG-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# NORMAL — CORRECTLY DETECTED
# ============================================================

a1nt_normal_correct = get_a1nt_outcome_subset(
    family="normal",
    outcome="correct",
)

inspect_a1nt_subset(
    a1nt_normal_correct,
    (
        "A1-NT — NORMAL CASES CORRECTLY "
        "PREDICTED AS NORMAL"
    ),
)


# ============================================================
# NORMAL — FALSELY FLAGGED
# ============================================================

a1nt_normal_missed = get_a1nt_outcome_subset(
    family="normal",
    outcome="missed",
)

inspect_a1nt_subset(
    a1nt_normal_missed,
    (
        "A1-NT — NORMAL CASES INCORRECTLY "
        "PREDICTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — CORRECTLY DETECTED
# ============================================================

a1nt_silent_partner_correct = get_a1nt_outcome_subset(
    family="silent_partner",
    outcome="correct",
)

inspect_a1nt_subset(
    a1nt_silent_partner_correct,
    (
        "A1-NT — SILENT-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — MISSED
# ============================================================

a1nt_silent_partner_missed = get_a1nt_outcome_subset(
    family="silent_partner",
    outcome="missed",
)

inspect_a1nt_subset(
    a1nt_silent_partner_missed,
    (
        "A1-NT — SILENT-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


A1-NT — LAG CASES CORRECTLY DETECTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,invalid_predictions,correct,accuracy_percent
0,90,0,90,0,90,0,90,100.0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_3sec,46,51.11
1,lag_2sec,44,48.89



ASSESSMENT BREAKDOWN


...

decisive_dimension,PARTICIPATION,All
semantic_assessment,,
COMPATIBLE,16,16
LIMITED,84,84
All,100,100



TEMPORAL × SEMANTIC ASSESSMENT


semantic_assessment,COMPATIBLE,LIMITED,All
temporal_assessment,,,
LIMITED,16,84,100
All,16,84,100



A1-NT — SILENT-PARTNER CASES NOT DETECTED (PREDICTED NORMAL)
No cases found in this subset.


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency


# Compare A0 Full Structured R1 with the three participation ablations

The cell loads the already-saved A0 metrics when they exist and places all configurations in one table.

In [ ]:

# ============================================================
# FINAL PARTICIPATION-ABLATION COMPARISON
#
# Loads the existing Full Structured R1 metrics when available,
# then appends the three new ablation results.
# ============================================================

def metrics_to_comparison_row(
    display_name,
    metrics,
):
    confusion = (
        metrics[
            "confusion_matrix"
        ]
    )

    return {
        "configuration": (
            display_name
        ),

        "valid_predictions": (
            metrics[
                "valid_predictions"
            ]
        ),

        "invalid_predictions": (
            metrics[
                "invalid_predictions"
            ]
        ),

        "gold_NORMAL_pred_NORMAL": int(
            confusion[0][0]
        ),

        "gold_NORMAL_pred_ANOMALOUS": int(
            confusion[0][1]
        ),

        "gold_ANOMALOUS_pred_NORMAL": int(
            confusion[1][0]
        ),

        "gold_ANOMALOUS_pred_ANOMALOUS": int(
            confusion[1][1]
        ),

        "accuracy": (
            metrics[
                "accuracy_valid_predictions"
            ]
        ),

        "balanced_accuracy": (
            metrics[
                "balanced_accuracy"
            ]
        ),

        "anomalous_f1": (
            metrics[
                "anomalous_f1"
            ]
        ),

        "normal_recall": (
            metrics[
                "normal_recall_specificity"
            ]
        ),
    }


comparison_rows = []


existing_r1_metrics_path = (
    OUT_DIR
    / (
        "reasoning_r1_full_semantics_"
        "normal_references_only"
    )
    / "metrics.json"
)


if existing_r1_metrics_path.exists():
    existing_r1_metrics = json.loads(
        existing_r1_metrics_path.read_text(
            encoding="utf-8"
        )
    )

    comparison_rows.append(
        metrics_to_comparison_row(
            "A0 — Full Structured R1",
            existing_r1_metrics,
        )
    )

else:
    print(
        "Existing Full Structured R1 metrics not found:",
        existing_r1_metrics_path,
    )


for display_name, evaluation in [
    (
        "A1 — No Participation Branch",
        A1_NO_PARTICIPATION_EVALUATION,
    ),

    (
        "A1-S — No Speaks, Keep Turns",
        A1_NO_SPEAKS_EVALUATION,
    ),

    (
        "A1-T — Keep Speaks, No Turns",
        A1_NO_TURNS_EVALUATION,
    ),
]:
    comparison_rows.append(
        metrics_to_comparison_row(
            display_name,
            evaluation[
                "metrics"
            ],
        )
    )


PARTICIPATION_ABLATION_COMPARISON = (
    pd.DataFrame(
        comparison_rows
    )
)


print("=" * 100)
print("PARTICIPATION-BRANCH ABLATION COMPARISON")
print("=" * 100)

display(
    PARTICIPATION_ABLATION_COMPARISON
)


comparison_output_path = (
    OUT_DIR
    / "participation_branch_ablation_comparison.csv"
)


PARTICIPATION_ABLATION_COMPARISON.to_csv(
    comparison_output_path,
    index=False,
)


print(
    "Saved comparison:",
    comparison_output_path,
)


PARTICIPATION-BRANCH ABLATION COMPARISON


,configuration,valid_predictions,invalid_predictions,gold_NORMAL_pred_NORMAL,gold_NORMAL_pred_ANOMALOUS,gold_ANOMALOUS_pred_NORMAL,gold_ANOMALOUS_pred_ANOMALOUS,accuracy,balanced_accuracy,anomalous_f1,normal_recall
0,A0 — Full Structured R1,400,0,78,22,32,268,0.8650,0.836667,0.908475,0.78
1,A1 — No Participation Branch,400,0,47,53,44,256,0.7575,0.661667,0.840722,0.47
2,"A1-S — No Speaks, Keep Turns",400,0,75,25,24,276,0.8775,0.835000,0.918469,0.75
3,"A1-T — Keep Speaks, No Turns",400,0,78,22,21,279,0.8925,0.855000,0.928453,0.78


Saved comparison: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/participation_branch_ablation_comparison.csv


# Optional: disconnect the Colab runtime

Run only after all three experiments and evaluations have completed.

In [ ]:
from google.colab import runtime
runtime.unassign()